# Validación cruzada sin agrupar efectores ni proteínas

# Nested CV con splits Cx explícitos — cuatro escenarios de generalización

Pipeline de evaluación de regresión logística + embeddings AlphaFold siguiendo los **cuatro escenarios de Park & Marcotte (2012, *Nature Methods*)** definidos en el esquema de selección de negativos:

| Escenario | Descripción | Generalización evaluada |
|-----------|-------------|------------------------|
| **C1** | Efector y proteína ya vistos en train | Más optimista (baseline) |
| **C2E** | Efector nuevo, proteína vista | Generalización sobre efectores |
| **C2P** | Proteína nueva, efector visto | Generalización sobre proteínas |
| **C3** | Ambos nuevos | Más exigente — uso real en inferencia |

Los splits de C2E, C2P y C3 se cargan directamente desde `generate_cx_splits.py` (evitando recomputar y garantizando reproducibilidad). C1 usa `StratifiedKFold` estándar como baseline sin restricción de leakage.

Al final del notebook se entrena un **modelo final** con todos los datos etiquetados y se realiza **inferencia sobre los casos dudosos**, indicando el nivel de confianza de cada predicción (C1/C2E/C2P/C3) según si el modelo ha visto moléculas similares durante el entrenamiento.

In [1]:
import sys
import json
import time
import warnings
from collections import defaultdict
from pathlib import Path

import numpy as np
import pandas as pd

from sklearn.pipeline import make_pipeline
from imblearn.pipeline import make_pipeline as make_pipeline_imb
from imblearn.over_sampling import RandomOverSampler
from sklearn.neural_network import MLPClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.utils.class_weight import compute_sample_weight
from sklearn.model_selection import (
    GridSearchCV, cross_validate,
    BaseCrossValidator, StratifiedKFold,
)

# Generador de splits Cx (debe estar en el mismo directorio o en PYTHONPATH)
sys.path.insert(0, str(Path("/home/jovyan/TFG").resolve()))
from generate_cx_splits import build_and_save_splits, load_splits

In [2]:
# ── Paths ─────────────────────────────────────────────────────────────────────
OUTPUT_DIR      = Path("/home/jovyan/TFG/output_Efectores_alphafold_all")
OUTPUT_RESULTS = Path("/home/jovyan/TFG/CV_Kmeans_con_EspS_NleK_all_models/results")
# PATH_LABELED    = OUTPUT_DIR / "labeled_interactions"    # pares etiquetados (pos + neg)
# PATH_UNKNOWN    = OUTPUT_DIR / "unknown_interactions"    # pares dudosos para inferencia
PATH_METADATA   = OUTPUT_RESULTS / "metadata.csv"           # sample_name, effector_group, protein_group, label
SPLITS_DIR      = OUTPUT_RESULTS / "splits"                 # directorio donde se guardan/leen los splits Cx

# Tipo de embedding y pooling por defecto (se sobreescribe tras la búsqueda)
DEFAULT_EMB_TYPE = "single_embeddings"

Preparamos los Data Frames para el entrenamiento y la inferencia que se usarán a continuación.

In [3]:
df_total = pd.read_csv("/home/jovyan/TFG/Interacciones_EffectorProteina_LiteratureOnly_Ordenadas_NleG.csv")
# Añadimos los grupos de efectores y proteínas
effector_groups = pd.read_csv("/home/jovyan/TFG/CV_Kmeans/effector_groups_kmeans.csv")
mapping_ef_groups = effector_groups.set_index("Effector")["Kmeans_Group"]
df_total["Effector_Group"] = df_total["Effector"].map(mapping_ef_groups)
protein_groups = pd.read_csv("/home/jovyan/TFG/CV_Kmeans/protein_groups_kmeans.csv")
mapping_prot_groups = protein_groups.set_index("Protein")["Kmeans_Group"]
df_total["Protein_Group"] = df_total["Protein"].map(mapping_prot_groups)
# Añadimos además una columna Sample name con el prefijo de las carpetas generadas por AlphaFold3
df_total["Sample_name"] = df_total["Protein"] + "_" + df_total["Effector"]
df_total.head()

,Effector,Protein,ProteinGeneName,Shared_Connections,Shortest_Path,Is_Connected,Effector_Group,Protein_Group,Sample_name
0,EspL,O89110,Casp8,4,1.0,True,Cluster_0,Cluster_1,O89110_EspL
1,EspL,Q60855,Ripk1,3,1.0,True,Cluster_0,Cluster_4,Q60855_EspL
2,NleB,O89110,Casp8,2,1.0,True,Cluster_0,Cluster_1,O89110_NleB
3,NleA,Q8R4B8,Nlrp3,1,1.0,True,Cluster_5,Cluster_4,Q8R4B8_NleA
4,NleA,Q9D8T2,Gsdmd,1,1.0,True,Cluster_5,Cluster_1,Q9D8T2_NleA


In [4]:
# Modificamos el Data Frame para que tenga solo la información que nos interesa
df_total = df_total.drop(columns=['Protein', 'Shared_Connections', 'Shortest_Path'])
df_total.rename(columns={'ProteinGeneName': 'Protein'}, inplace=True)
df_total.head()

,Effector,Protein,Is_Connected,Effector_Group,Protein_Group,Sample_name
0,EspL,Casp8,True,Cluster_0,Cluster_1,O89110_EspL
1,EspL,Ripk1,True,Cluster_0,Cluster_4,Q60855_EspL
2,NleB,Casp8,True,Cluster_0,Cluster_1,O89110_NleB
3,NleA,Nlrp3,True,Cluster_5,Cluster_4,Q8R4B8_NleA
4,NleA,Gsdmd,True,Cluster_5,Cluster_1,Q9D8T2_NleA


#### Pulido de las carpetas de output

En el output de AlphaFold3 hay carpetas que no fue capaz de ejecutar, con lo que se quedaron a medias. Esas carpetas nos interesa quitarlas del entrenamiento y de la inferencia porque no podemos trabajar con esos datos.

Localizamos qué muestras son y las quitamos del análisis.

In [5]:
incomplete_samples = set()

for sample in OUTPUT_DIR.iterdir():
    if sample.is_dir() and ".ipynb_checkpoints" not in sample.name:
        if not (sample / "TERMS_OF_USE.md") in sample.iterdir():
            # Nos quedamos con el nombre limpio y lo guardamos en la lista
            parts = sample.name.split('_')
            clean_name = "_".join(parts[:2])
            incomplete_samples.add(clean_name)

len(incomplete_samples)
incomplete_samples

{'B2RWS6_EspN',
 'B2RWS6_NleL',
 'P26039_EspL',
 'P26039_EspN',
 'P26039_NleL',
 'P26039_Tir'}

In [6]:
# Quitamos todas esas parejas que no ha conseguido procesar de la lista de interacciones totales y nos olvidamos de ellas
df_total = df_total[~df_total["Sample_name"].isin(incomplete_samples)]
len(df_total)

5466

In [7]:
# Creamos un Data Frame para las interacciones que formarán parte del train
# Añadimos una columna de Sample name que coincidirá con el prefijo de la carpeta generada por AlphaFold3
df_labeled = pd.read_csv("Interacciones_Entrenamiento_CV_estricto_kmeans_con_EspS_NleK.csv")
uniprot_equivalences = pd.read_csv("/home/jovyan/TFG/Interacciones_EffectorProteina_LiteratureOnly_Ordenadas_NleG.csv", sep=",")
prot_id = pd.Series(uniprot_equivalences.Protein.values, index=uniprot_equivalences.ProteinGeneName).to_dict()
df_labeled["Protein_ID"] = df_labeled["Protein"].map(prot_id)
df_labeled["Sample_name"] = df_labeled["Protein_ID"] + "_" + df_labeled["Effector"]
# y nos quedamos con las mismas columnas que en el Data Frame total
df_labeled = df_labeled.drop(columns=['Pathways_Shared', 'Shared_Connections', 'Shortest_Path', 'Interaction_Score', 'Protein_ID'])
# Es necesario además que quitemos de df_labeled también las muestras incomplete_samples
# por la misma razón de antes
df_labeled = df_labeled[~df_labeled["Sample_name"].isin(incomplete_samples)]
df_labeled.head()

,Effector,Protein,Is_Connected,Effector_Group,Protein_Group,Label,Sample_name
0,EspM,Rhoc,True,Cluster_6,Cluster_3,1,Q62159_EspM
1,NleB,Ripk1,True,Cluster_0,Cluster_4,1,Q60855_NleB
2,NleB,Nfkb1,True,Cluster_0,Cluster_4,1,P25799_NleB
3,EspH,Rac2,True,Cluster_1,Cluster_3,1,Q05144_EspH
4,NleD,Mapkapk2,True,Cluster_6,Cluster_1,1,P49138_NleD


In [8]:
# Repetimos ahora con las interacciones desconocidas que se usarán en la inferencia
train_samples = set(df_labeled["Sample_name"])
df_unknown = df_total[~df_total["Sample_name"].isin(train_samples)]
df_unknown.head()

,Effector,Protein,Is_Connected,Effector_Group,Protein_Group,Sample_name
25,NleD,Mapk14,True,Cluster_6,Cluster_3,P47811_NleD
46,EspF,Casp3,True,Cluster_2,Cluster_3,O08668_EspF
47,EspF,Casp6,True,Cluster_2,Cluster_3,O08738_EspF
48,EspF,Casp7,True,Cluster_2,Cluster_3,O08669_EspF
82,Tir,Casp4,True,Cluster_2,Cluster_1,P70343_Tir


In [9]:
# Comprobamos que las longitudes son correctas
print(f"Parejas etiquetadas: {len(df_labeled)}")
print(f"Parejas cuya interacción se desconoce: {len(df_unknown)}")
print(f"Parejas totales: {len(df_total)}")
print(f"Suma de parejas etiquetadas y desconocidas: {len(df_labeled)} + {len(df_unknown)} = {len(df_labeled) + len(df_unknown)}")

Parejas etiquetadas: 962
Parejas cuya interacción se desconoce: 4504
Parejas totales: 5466
Suma de parejas etiquetadas y desconocidas: 962 + 4504 = 5466


In [10]:
# Por último añadimos una columna de etiquetas a las parejas que se van a usar en el entrenamiento para saber si son positivas o negativas
df_labeled["Label"] = df_labeled["Is_Connected"].astype(int)
df_unknown["Label"] = "-"
df_total["Label"] = df_total.apply(
    lambda x: int(x["Is_Connected"]) if x["Sample_name"] in train_samples else "-",
    axis=1
)

# df_total.to_csv("Interacciones_totales_CV.csv")
# df_labeled.to_csv("Interacciones_entrenamiento_relajado_CV.csv")
# df_unknown.to_csv("Interacciones_inferencia_relajado_CV.csv")

df_labeled.head()

,Effector,Protein,Is_Connected,Effector_Group,Protein_Group,Label,Sample_name
0,EspM,Rhoc,True,Cluster_6,Cluster_3,1,Q62159_EspM
1,NleB,Ripk1,True,Cluster_0,Cluster_4,1,Q60855_NleB
2,NleB,Nfkb1,True,Cluster_0,Cluster_4,1,P25799_NleB
3,EspH,Rac2,True,Cluster_1,Cluster_3,1,Q05144_EspH
4,NleD,Mapkapk2,True,Cluster_6,Cluster_1,1,P49138_NleD


# Funciones

### 1. Carga de embeddings

Funciones para cargar embeddings de AlphaFold en *chunks* y construir bloques
(X, y, sample_names) para el entrenamiento. La función `load_block` también devuelve
los `sample_names` necesarios para mapear a los splits Cx.

In [11]:
# def load_samples_in_chunks(input_dir, batch_size=2, emb_type="single_embeddings", transforms=None):
#     """
#     Generator que carga embeddings y etiquetas en batches para eficiencia de memoria.

#     :param input_dir:   Directorio raíz con las carpetas de cada muestra.
#     :param batch_size:  Número de muestras por batch.
#     :param emb_type:    Clave del array dentro del .npz (p.ej. 'single_embeddings').
#     :param transforms:  Lista opcional de funciones a aplicar al embedding cargado.
#     :yields: Tuple (np.ndarray de embeddings del batch, lista de etiquetas).
#     """
#     def load_sample(name):
#         path = input_dir / name / "seed-0_embeddings" / (name + "_seed-0_embeddings.npz")
#         embeddings = np.load(path)
#         label = int(name.split("_")[-1])
#         return embeddings[emb_type], label

#     sample_names = sorted([d.name for d in input_dir.iterdir() if d.is_dir()])

#     for i in range(0, len(sample_names), batch_size):
#         batch = sample_names[i:i + batch_size]
#         out, labels = [], []
#         for name in batch:
#             repr_emb, label = load_sample(name)
#             if transforms is None:
#                 out.append(repr_emb)
#             elif len(transforms) == 1:
#                 out.append(transforms[0](repr_emb))
#             else:
#                 out.append([t(repr_emb) for t in transforms])
#             labels.append(label)
#         yield np.asarray(out), labels


# def load_block(input_dir, emb_type="single_embeddings", transforms=None):
#     """
#     Carga todos los embeddings y etiquetas de un directorio completo.
#     Devuelve también sample_names (necesarios para alinear con splits Cx).

#     :returns: X (np.ndarray), y (np.ndarray), sample_names (list[str])
#     """
#     X, y = [], []
#     sample_names = sorted([d.name for d in input_dir.iterdir() if d.is_dir()])

#     for sample, labels in load_samples_in_chunks(
#         input_dir, emb_type=emb_type, transforms=transforms
#     ):
#         X.append(sample)
#         y.extend(labels)

#     return np.concatenate(X, axis=0), np.array(y), sample_names

In [12]:
# def load_samples_in_chunks_no_label(input_dir, batch_size=2, emb_type="single_embeddings", transforms=None):
#     """Generator igual a load_samples_in_chunks pero sin etiquetas (para dudosos)."""
#     def load_sample(name):
#         path = input_dir / name / "seed-0_embeddings" / (name + "_seed-0_embeddings.npz")
#         return np.load(path)[emb_type]

#     sample_names = sorted([d.name for d in input_dir.iterdir() if d.is_dir()])
#     for i in range(0, len(sample_names), batch_size):
#         batch = sample_names[i:i + batch_size]
#         out = []
#         for name in batch:
#             repr_emb = load_sample(name)
#             if transforms is None:
#                 out.append(repr_emb)
#             elif len(transforms) == 1:
#                 out.append(transforms[0](repr_emb))
#             else:
#                 out.append([t(repr_emb) for t in transforms])
#         yield np.asarray(out)


# def load_block_no_label(input_dir, emb_type="single_embeddings", transforms=None):
#     """Carga embeddings sin etiquetas (casos dudosos para inferencia).

#     :returns: X (np.ndarray), sample_names (list[str])
#     """
#     X = []
#     sample_names = sorted([d.name for d in input_dir.iterdir() if d.is_dir()])
#     for sample in load_samples_in_chunks_no_label(
#         input_dir, emb_type=emb_type, transforms=transforms
#     ):
#         X.append(sample)
#     return np.concatenate(X, axis=0), sample_names

Funciones para cargar embeddings de AlphaFold de acuerdo a las muestras presentes en el Data Frame proporcionado.

In [13]:
def load_block_from_csv(input_dir, target_df, emb_type="single_embeddings", transforms=None):
    import numpy as np
    X, y, sample_names = [], [], []

    for _, row in target_df.iterrows():
        sample_id = str(row['Sample_name']).strip()

        # --- CAMBIO AQUÍ: Buscar carpetas que EMPIECEN por el ID ---
        # Esto encontrará "ID", "ID_0", "ID_1", etc.
        matching_folders = list(input_dir.glob(f"{sample_id}*"))
        # Filtrar para asegurarnos de que sean directorios
        matching_folders = [f for f in matching_folders if f.is_dir()]

        if matching_folders:
            # Tomamos la primera coincidencia encontrada
            folder_path = matching_folders[0]

            # Buscamos el .npz dentro
            npz_folder = folder_path / "seed-0_embeddings"
            npz_files = list(npz_folder.glob("*.npz")) if npz_folder.exists() else []

            if npz_files:
                try:
                    path = npz_files[0]
                    emb_data = np.load(path)[emb_type]

                    if transforms is None:
                        X.append(emb_data)
                    elif len(transforms) == 1:
                        X.append(transforms[0](emb_data))
                    else:
                        X.append([t(emb_data) for t in transforms])

                    sample_names.append(sample_id)
                    if 'Label' in target_df.columns:
                        try:
                            # Intentamos convertir la etiqueta a entero (para entrenamiento)
                            y.append(int(row['Label']))
                        except (ValueError, TypeError):
                            # Si es un guion "-" o NaN (modo inferencia), añadimos un valor nulo o None
                            y.append(None)

                except Exception as e:
                    print(f"❌ Error al cargar datos de {sample_id}: {e}")
        else:
            # print(f"❓ No se encontró carpeta para: {sample_id}")
            pass

    X_array = np.asarray(X)
    if X_array.ndim > 2 and transforms and len(transforms) > 1:
        X_array = np.swapaxes(X_array, 0, 1)

    print(f"✅ Éxito: Se cargaron {len(X)} muestras de las {len(target_df)} del Excel.")
    return X_array, (np.array(y) if y else None), sample_names

### 2. Splits Cx — generación y carga

`build_and_save_splits` (de `generate_cx_splits.py`) genera y serializa los splits
C3/C2E/C2P a disco. `load_splits` los recupera.

`PrecomputedSplitter` traduce los splits (listas de `sample_name`) a índices de array,
haciéndolos compatibles con `cross_validate` y `GridSearchCV` de scikit-learn.

In [14]:
class PrecomputedSplitter(BaseCrossValidator):
    """
    Splitter de scikit-learn que usa splits precalculados (de generate_cx_splits).

    Traduce listas de sample_name a índices de numpy para compatibilidad
    con GridSearchCV y el bucle externo manual de hpm_search_nested_cv.
    """

    def __init__(self, folds: dict, sample_names: list):
        self.folds = folds
        self.name2idx = {name: i for i, name in enumerate(sample_names)}

    def _resolve(self, names):
        """Convierte lista de sample_name a array de índices (ignorando ausentes)."""
        return np.array(
            [self.name2idx[n] for n in names if n in self.name2idx],
            dtype=int,
        )

    def _iter_test_indices(self, X=None, y=None, groups=None):
        for fold in self.folds.values():
            yield self._resolve(fold["test"])

    def split(self, X, y=None, groups=None):
        for fold in self.folds.values():
            train_idx = self._resolve(fold["train"])
            test_idx  = self._resolve(fold["test"])
            yield train_idx, test_idx

    def get_n_splits(self, X=None, y=None, groups=None):
        return len(self.folds)

    def _iter_test_masks(self, X=None, y=None, groups=None):
        n = X.shape[0] if X is not None else len(self.name2idx)
        for test_idx in self._iter_test_indices(X, y, groups):
            mask = np.zeros(n, dtype=bool)
            if len(test_idx):
                mask[test_idx] = True
            yield mask


def get_outer_splitter(scenario: str, sample_names: list, splits_dir) -> BaseCrossValidator:
    """
    Devuelve el splitter externo apropiado para cada escenario:
      C1  → StratifiedKFold(5) — sin restricción de leakage (baseline).
      C2E → leave-one-effector_group-out  (precomputado).
      C2P → leave-one-protein_group-out   (precomputado).
      C3  → leave-one-(eg × pg)-out       (precomputado).
    """
    if scenario == "C1":
        return StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    folds = load_splits(str(splits_dir), scenario)
    return PrecomputedSplitter(folds, sample_names)


def get_inner_groups(groups_eg: np.ndarray, groups_pg: np.ndarray, scenario: str) -> np.ndarray:
    """
    Construye el array de grupos para el bucle INTERNO (GridSearchCV) a partir
    de los grupos del subconjunto de train del fold externo.

    El inner loop debe respetar la misma restricción que el outer para no
    introducir leakage en la selección de hiperparámetros
    (Krstajic et al., 2014, J Cheminform; Varoquaux et al., 2017, NeuroImage).

    Mapeo por escenario:
      C1  → None           (StratifiedKFold, sin grupos).
      C2E → groups_eg      (leave-one-effector_group-out en el inner train).
      C2P → groups_pg      (leave-one-protein_group-out en el inner train).
      C3  → eg + '__' + pg (leave-one-(eg × pg)-out en el inner train).
    """
    if scenario == "C1":
        return None
    if scenario == "C2E":
        return groups_eg
    if scenario == "C2P":
        return groups_pg
    if scenario == "C3":
        return np.array([f"{eg}__{pg}" for eg, pg in zip(groups_eg, groups_pg)])
    raise ValueError(f"Escenario desconocido: {scenario}")


def get_inner_splitter(scenario: str, inner_groups: np.ndarray, inner_cv: int = 5):
    """
    Devuelve (splitter_inner, groups_para_fit) para el GridSearchCV interno.

    Para C2E/C2P/C3 se usa LeaveOneGroupOut con los grupos del subconjunto de
    train, garantizando que la búsqueda de hiperparámetros no ve combinaciones
    no vistas en el fold de train externo.

    Para C1 se usa StratifiedKFold sin grupos.
    """
    from sklearn.model_selection import LeaveOneGroupOut
    if scenario == "C1":
        return StratifiedKFold(n_splits=inner_cv, shuffle=True, random_state=42), None
    return LeaveOneGroupOut(), inner_groups


### 3. Verificación de splits

Antes de entrenar, se imprime el resumen de cada fold: tamaño de train/test,
clases presentes y (para C3) muestras excluidas. Los folds con una sola clase
producirán NaN en las métricas durante el CV; se contabilizan y se informa.

In [15]:
def verify_cx_splits(scenario: str, splitter, X, y, sample_names, splits_dir):
    """
    Comprueba que PrecomputedSplitter ha cargado exactamente las particiones
    generadas por generate_cx_splits, comparando contra los metadatos JSON.

    Solo imprime el resumen final y los folds que no cuadren (si los hay).
    El reporte completo fold a fold ya está en splits_report.txt.

    :returns: (n_valid, n_one_class, n_empty)
    """
    import json as _json
    from pathlib import Path as _Path

    # C1 no tiene metadatos Cx — solo contar folds válidos
    if scenario == "C1":
        n_valid = n_one_class = n_empty = 0
        for train_idx, test_idx in splitter.split(X, y):
            if len(test_idx) == 0:
                n_empty += 1
            elif len(set(y[test_idx])) < 2:
                n_one_class += 1
            else:
                n_valid += 1
        viab = "✅ viable" if n_valid >= 3 else "❌ inviable"
        print(f"  [{scenario}] {splitter.get_n_splits(X, y)} folds  "
              f"válidos={n_valid}  una clase={n_one_class}  vacíos={n_empty}  {viab}")
        return n_valid, n_one_class, n_empty

    # Cargar metadatos (fuente de verdad de generate_cx_splits)
    meta_path = _Path(splits_dir) / f"splits_{scenario}_meta.json"
    with open(meta_path) as f:
        meta = _json.load(f)

    fold_labels = list(splitter.folds.keys())
    n_valid = n_one_class = n_empty = mismatches = 0
    mismatch_lines = []

    for label, (train_idx, test_idx) in zip(fold_labels, splitter.split(X, y)):
        if len(test_idx) == 0:
            n_empty += 1
            continue

        pos = int(y[test_idx].sum())
        neg = int(len(test_idx) - pos)

        if pos == 0 or neg == 0:
            n_one_class += 1
        else:
            n_valid += 1

        m = meta.get(str(label), {})
        errors = []
        if m.get("n_test_pos") != pos:
            errors.append(f"pos: got {pos} expected {m.get('n_test_pos')}")
        if m.get("n_test_neg") != neg:
            errors.append(f"neg: got {neg} expected {m.get('n_test_neg')}")
        if m.get("n_train") != len(train_idx):
            errors.append(f"train: got {len(train_idx)} expected {m.get('n_train')}")
        if errors:
            mismatches += 1
            mismatch_lines.append(f"    ❌ fold [{label}]: {', '.join(errors)}")

    viab = "✅ viable" if n_valid >= 3 else "❌ inviable"
    cross = "✅ coincide con report_splits" if mismatches == 0             else f"❌ {mismatches} fold(s) NO coinciden con report_splits"
    print(f"  [{scenario}] {len(fold_labels)} folds  "
          f"válidos={n_valid}  una clase={n_one_class}  vacíos={n_empty}  "
          f"{viab}  |  {cross}")
    for line in mismatch_lines:
        print(line)

    return n_valid, n_one_class, n_empty


### 4. Nested Cross-Validation multi-escenario

`hpm_search_nested_cv` es la función central: bucle externo con el splitter del
escenario elegido y bucle interno con `StratifiedKFold` para la búsqueda de
hiperparámetros. Los folds vacíos o de una clase retornan NaN (gestionados por
`np.nanmean` en el análisis posterior).

`test_pooling_transforms_all_scenarios` itera sobre todos los tipos de embedding,
poolings y escenarios en un único paso, devolviendo un diccionario
`results[scenario][emb_name]`.

In [16]:
def hpm_search_nested_cv(
    X, y,
    outer_splitter,
    groups_eg: np.ndarray,
    groups_pg: np.ndarray,
    scenario: str,
    inner_cv: int = 5,
):
    """
    Nested CV con bucle EXTERNO manual y bucle INTERNO consistente con el escenario.

    El bucle externo se implementa manualmente (sin cross_validate) para poder
    construir en cada fold los groups internos restringidos al subconjunto de train.
    Esto es necesario porque cross_validate no soporta groups dinámicos por fold.

    Bucle externo : outer_splitter (PrecomputedSplitter o StratifiedKFold).
    Bucle interno : LeaveOneGroupOut con grupos del subconjunto de train
                    para C2E / C2P / C3; StratifiedKFold para C1.

    Importante: usar el mismo tipo de split en ambos bucles es el requisito
    estándar para nested CV con datos agrupados (Krstajic et al., 2014,
    J Cheminform; Varoquaux et al., 2017, NeuroImage).

    :param X:             Array de embeddings (n_samples, n_features).
    :param y:             Array de etiquetas.
    :param outer_splitter: BaseCrossValidator para el bucle externo.
    :param groups_eg:     Array de grupos de efector (len = n_samples).
    :param groups_pg:     Array de grupos de proteína (len = n_samples).
    :param scenario:      'C1', 'C2E', 'C2P' o 'C3'.
    :param inner_cv:      n_splits para StratifiedKFold en C1.
    :returns: Dict con claves 'test_accuracy', 'test_roc_auc', 'test_pr_auc',
              'estimator' (lista de GridSearchCV ajustados por fold).
    """
    from sklearn.model_selection import LeaveOneGroupOut, GridSearchCV
    from sklearn.metrics import (balanced_accuracy_score, roc_auc_score, average_precision_score, matthews_corrcoef, precision_recall_curve, f1_score, precision_score, recall_score)
    
    param_grid = [
        {
            "mlpclassifier__hidden_layer_sizes": [(256, 128, 64), (128, 64, 32), (512, 256, 128)],
            "mlpclassifier__alpha":              [0.0001, 0.001, 0.01],
            "mlpclassifier__learning_rate_init": [0.001, 0.0001],
        }
    ]

    pipeline = make_pipeline_imb(
        StandardScaler(),
        RandomOverSampler(random_state=42), # Para solucionar el desbalanceo en MLP
        MLPClassifier(
            activation='relu',
            solver='adam',
            max_iter=500,
            early_stopping=False,   # evita reducir aún más sets de train muy pequeños (C3)
            random_state=42,
        )
    )

    metrics_keys = (
        "test_bal_accuracy_50", "test_mcc_50", "test_precision_50", "test_recall_50", "test_pr_gain_50", "test_f1g_50",
        "test_bal_accuracy_opt", "test_mcc_opt", "test_precision_opt", "test_recall_opt", "test_pr_gain_opt", "test_f1g_opt",
        "test_best_threshold", "test_roc_auc", "test_pr_auc", "test_lift", "test_auprg", "test_expected_f1g"
    )
    
    results = {k: [] for k in metrics_keys}
    results["estimator"] = []
    results["fold_details"] = []

    # Extraer fold_ids si el splitter es PrecomputedSplitter (Cx); usar índice para C1
    if hasattr(outer_splitter, "folds"):
        fold_ids = list(outer_splitter.folds.keys())
    else:
        fold_ids = [f"fold_{i}" for i in range(outer_splitter.get_n_splits(X, y))]

    for fold_id, (train_idx, test_idx) in zip(fold_ids, outer_splitter.split(X, y)):
        # ── Fold vacío o de una sola clase → NaN ─────────────────────────────
        if len(test_idx) == 0:
            for k in ("test_bal_accuracy", "test_mcc", "test_roc_auc", "test_pr_auc"):
                results[k].append(np.nan)
                results["estimator"].append(None)
                results["fold_details"].append({"fold_id": fold_id, "n_test": 0,
                "n_test_pos": 0, "n_test_neg": 0,
                "bal_accuracy": np.nan, "mcc": np.nan, "roc_auc": np.nan, "pr_auc": np.nan,
                "best_hidden_layer_sizes": "", "best_alpha": np.nan, "best_lr_init": np.nan, "note": "vacío"})
            continue

        y_test = y[test_idx]
        
        if len(np.unique(y_test)) < 2:
            for k in metrics_keys: results[k].append(np.nan)
            results["estimator"].append(None)
            results["fold_details"].append({"fold_id": fold_id, "note": "una clase"})
            continue

        X_train, y_train = X[train_idx], y[train_idx]
        X_test            = X[test_idx]

        # ── Construir grupos del subconjunto de train para el inner CV ────────
        eg_train = groups_eg[train_idx]
        pg_train = groups_pg[train_idx]
        inner_groups = get_inner_groups(eg_train, pg_train, scenario)
        inner_splitter, fit_groups = get_inner_splitter(scenario, inner_groups, inner_cv)

        # ── Inner CV: búsqueda de hiperparámetros ────────────────────────────
        grid = GridSearchCV(
            pipeline, param_grid,
            cv=inner_splitter,
            scoring="average_precision",   # PR AUC como criterio de selección
            n_jobs=-1,
            error_score=np.nan,
        )
        grid.fit(X_train, y_train, groups=fit_groups)

        # ── Evaluación en test externo ────────────────────────────────────────
        try:
            proba = grid.predict_proba(X_test)[:, 1]
            pi = float(y_test.sum() / len(y_test)) # Prevalencia de positivos

            # 1. EVALUACIÓN CON UMBRAL FIJO (0.50)
            pred_50 = (proba >= 0.5).astype(int)
            bal_acc_50 = balanced_accuracy_score(y_test, pred_50)
            mcc_50     = matthews_corrcoef(y_test, pred_50)
            prec_50    = precision_score(y_test, pred_50, zero_division=0)
            rec_50     = recall_score(y_test, pred_50, zero_division=0)
            f1_50      = f1_score(y_test, pred_50, zero_division=0)
            
            pr_gain_50 = (prec_50 - pi) / ((1.0 - pi) * prec_50) if prec_50 > pi else 0.0
            f1g_50     = (f1_50 - pi) / ((1.0 - pi) * f1_50) if f1_50 > pi else 0.0

            # 2. BARRIDO DE UMBRALES PARA ENCONTRAR EL MEJOR (MAX F1)
            thresholds = np.linspace(0.01, 0.99, 100)
            best_f1 = -1.0
            best_thresh = 0.5
            
            for t in thresholds:
                current_pred = (proba >= t).astype(int)
                current_f1 = f1_score(y_test, current_pred, zero_division=0)
                if current_f1 > best_f1:
                    best_f1 = current_f1
                    best_thresh = t

            # 3. EVALUACIÓN CON UMBRAL ÓPTIMO
            pred_opt = (proba >= best_thresh).astype(int)
            bal_acc_opt = balanced_accuracy_score(y_test, pred_opt)
            mcc_opt     = matthews_corrcoef(y_test, pred_opt)
            prec_opt    = precision_score(y_test, pred_opt, zero_division=0)
            rec_opt     = recall_score(y_test, pred_opt, zero_division=0)
            f1_opt      = best_f1

            pr_gain_opt = (prec_opt - pi) / ((1.0 - pi) * prec_opt) if prec_opt > pi else 0.0
            f1g_opt     = (f1_opt - pi) / ((1.0 - pi) * f1_opt) if f1_opt > pi else 0.0

            # 4. CURVA COMPLETA E INTEGRACIÓN GEOMÉTRICA (AUPRG y E[FG1])
            roc     = roc_auc_score(y_test, proba)
            pr      = average_precision_score(y_test, proba)
            lift    = pr / pi if pi > 0 else np.nan

            precisions, recalls, _ = precision_recall_curve(y_test, proba)
            with np.errstate(divide='ignore', invalid='ignore'):
                prec_g_curve = (precisions - pi) / ((1.0 - pi) * precisions)
                rec_g_curve = (recalls - pi) / ((1.0 - pi) * recalls)

            prec_g_curve = np.clip(np.nan_to_num(prec_g_curve, nan=0.0), 0.0, 1.0)
            rec_g_curve = np.clip(np.nan_to_num(rec_g_curve, nan=0.0), 0.0, 1.0)

            sort_idx = np.argsort(rec_g_curve)
            rec_g_sorted = rec_g_curve[sort_idx]
            prec_g_sorted = prec_g_curve[sort_idx]
            
            auprg = float(np.trapz(prec_g_sorted, rec_g_sorted))

            # Extracción de y0 (Precision Gain cuando Recall Gain == 0)
            zero_rec_indices = np.where(rec_g_sorted == 0.0)[0]
            y0 = float(prec_g_sorted[zero_rec_indices[-1]]) if len(zero_rec_indices) > 0 else 0.0

            # Cálculo del F1-Gain Esperado 
            if y0 == 1.0:
                expected_f1g = auprg / 2.0 + 0.25
            else:
                num = auprg / 2.0 + 0.25 - (pi * (1.0 - y0**2)) / 4.0
                den = 1.0 - pi * (1.0 - y0)
                expected_f1g = float(num / den) if den != 0 else 0.0

            # Guardar en estructura general
            results["test_bal_accuracy_50"].append(bal_acc_50)
            results["test_mcc_50"].append(mcc_50)
            results["test_precision_50"].append(prec_50)
            results["test_recall_50"].append(rec_50)
            results["test_pr_gain_50"].append(pr_gain_50)
            results["test_f1g_50"].append(f1g_50)
            
            results["test_bal_accuracy_opt"].append(bal_acc_opt)
            results["test_mcc_opt"].append(mcc_opt)
            results["test_precision_opt"].append(prec_opt)
            results["test_recall_opt"].append(rec_opt)
            results["test_pr_gain_opt"].append(pr_gain_opt)
            results["test_f1g_opt"].append(f1g_opt)
            
            results["test_best_threshold"].append(best_thresh)
            results["test_roc_auc"].append(roc)
            results["test_pr_auc"].append(pr)
            results["test_lift"].append(lift)
            results["test_auprg"].append(auprg)
            results["test_expected_f1g"].append(expected_f1g)

            # Extraer hiperparámetros del mejor estimador
            bp = grid.best_params_ if hasattr(grid, "best_params_") else {}
            hls_key   = next((k for k in bp if k.endswith("__hidden_layer_sizes")), None)
            alpha_key = next((k for k in bp if k.endswith("__alpha")), None)
            lr_key    = next((k for k in bp if k.endswith("__learning_rate_init")), None)
            
            results["fold_details"].append({
                "fold_id":      fold_id,
                "n_test":       len(test_idx),
                "n_test_pos":   int(y_test.sum()),
                "n_test_neg":   int(len(test_idx) - y_test.sum()),
                "roc_auc":      round(roc, 4),
                "pr_auc":       round(pr, 4),
                "lift":         round(lift, 4) if not np.isnan(lift) else np.nan,
                "auprg":        round(auprg, 4),
                "expected_f1g": round(expected_f1g, 4),
                "best_threshold": round(best_thresh, 4),
                
                "bal_accuracy_50": round(bal_acc_50, 4),
                "mcc_50":          round(mcc_50, 4),
                "precision_50":    round(prec_50, 4),
                "recall_50":       round(rec_50, 4),
                "pr_gain_50":      round(pr_gain_50, 4),
                "f1g_50":          round(f1g_50, 4),
                
                "bal_accuracy_opt": round(bal_acc_opt, 4),
                "mcc_opt":          round(mcc_opt, 4),
                "precision_opt":    round(prec_opt, 4),
                "recall_opt":       round(rec_opt, 4),
                "pr_gain_opt":      round(pr_gain_opt, 4),
                "f1g_opt":          round(f1g_opt, 4),
                
                "best_hidden_layer_sizes": str(bp.get(hls_key, ""))   if hls_key   else "",
                "best_alpha":              bp.get(alpha_key, np.nan)  if alpha_key else np.nan,
                "best_lr_init":            bp.get(lr_key, np.nan)     if lr_key    else np.nan,
                "note":         "ok",
            })
            
        except Exception as e:
            for k in metrics_keys: results[k].append(np.nan)
            results["fold_details"].append({"fold_id": fold_id, "note": f"error: {str(e)}"})
            
        results["estimator"].append(grid)

    # ── Resumen de folds ──────────────────────────────────────────────────────
    acc_arr  = np.array(results["test_bal_accuracy_opt"], dtype=float)
    n_total  = len(acc_arr)
    n_valid  = int((~np.isnan(acc_arr)).sum())
    print(f"    folds: total={n_total}  válidos={n_valid}  NaN={n_total - n_valid}")

    return results



In [17]:
def test_pooling_transforms_all_scenarios(transforms, labeled_dir, splits_dir,
                                           metadata_df: pd.DataFrame,
                                           scenarios=None):
    """
    Evalúa todas las combinaciones (escenario × embedding × pooling) con Nested CV.

    El bucle interno de cada escenario usa la misma restricción de grupo que
    el bucle externo (C2E→efector, C2P→proteína, C3→celda, C1→estratificado).

    :param transforms:   Dict {emb_type: [lista de funciones de pooling]}.
    :param labeled_dir:  Path al directorio de muestras etiquetadas.
    :param splits_dir:   Path donde se guardaron los splits Cx.
    :param metadata_df:  DataFrame con columnas sample_name, effector_group, protein_group.
    :param scenarios:    Lista de escenarios a evaluar (por defecto los 4).
    :returns: Dict anidado all_results[scenario][emb_name] con métricas por pooling.
    """

    import time
    from collections import defaultdict
                                               
    if scenarios is None:
        scenarios = ["C1", "C2E", "C2P", "C3"]

    all_results = {}

    # Preconstruir mapas de grupo a partir de metadata_df
    eg_map = dict(zip(metadata_df["Sample_name"], metadata_df["Effector_Group"]))
    pg_map = dict(zip(metadata_df["Sample_name"], metadata_df["Protein_Group"]))

    for scenario in scenarios:
        print(f"\n{'#'*70}")
        print(f"#  ESCENARIO {scenario}")
        print(f"{'#'*70}")
        all_results[scenario] = {}

        for emb_name, transform_list in transforms.items():
            print(f"\n{'='*65}")
            print(f"  Embedding: {emb_name}  ({len(transform_list)} poolings)")
            print(f"{'='*65}")

            X_full, y_full, sample_names = load_block_from_csv(
                labeled_dir, target_df=metadata_df[metadata_df['Label'].isin([0,1])], emb_type=emb_name, transforms=transform_list
            )
            print(X_full, y_full)

            # Arrays de grupos por muestra — alineados con X_full
            groups_eg = np.array([eg_map.get(n, "UNKNOWN") for n in sample_names])
            groups_pg = np.array([pg_map.get(n, "UNKNOWN") for n in sample_names])

            # Splitter externo
            outer_splitter = get_outer_splitter(scenario, sample_names, splits_dir)

            # SI EL SHAPE ES (3, 394, 384), necesitamos mover el 394 al principio
            if X_full.ndim == 3 and X_full.shape[0] != len(y_full):
                # Esto cambia (3, 394, 384) -> (394, 3, 384)
                X_full = np.transpose(X_full, (1, 0, 2))

            # Verificar splits (consume el iterador; reconstruir después)
            verify_cx_splits(scenario, outer_splitter, X_full, y_full, sample_names, splits_dir=splits_dir)
            outer_splitter = get_outer_splitter(scenario, sample_names, splits_dir)

            res_emb = defaultdict(list)

            for i, _ in enumerate(transform_list):
                pooling_name = ["mean", "max", "min"][i] if i < 3 else f"pooling_{i}"
                print(f"\n  --- {emb_name} | {pooling_name} ---")

                Xi = X_full[:, i, :] if X_full.ndim == 3 else X_full

                # Reconstruir splitter externo para cada pooling
                outer_splitter = get_outer_splitter(scenario, sample_names, splits_dir)

                t0 = time.time()
                nested_res = hpm_search_nested_cv(
                    Xi, y_full,
                    outer_splitter=outer_splitter,
                    groups_eg=groups_eg,
                    groups_pg=groups_pg,
                    scenario=scenario,
                )
                elapsed = time.time() - t0

                # Extracción para estadísticas limpias
                bal50   = np.array(nested_res["test_bal_accuracy_50"], dtype=float)
                mcc50   = np.array(nested_res["test_mcc_50"],           dtype=float)
                pr50   = np.array(nested_res["test_precision_50"],     dtype=float)
                roc50   = np.array(nested_res["test_recall_50"],        dtype=float)
                prg50  = np.array(nested_res["test_pr_gain_50"],       dtype=float)
                f50   = np.array(nested_res["test_f1g_50"],           dtype=float)
                
                balopt  = np.array(nested_res["test_bal_accuracy_opt"], dtype=float)
                mccopt  = np.array(nested_res["test_mcc_opt"],          dtype=float)
                propt  = np.array(nested_res["test_precision_opt"],    dtype=float)
                rocopt  = np.array(nested_res["test_recall_opt"],       dtype=float)
                prgopt = np.array(nested_res["test_pr_gain_opt"],      dtype=float)
                fopt  = np.array(nested_res["test_f1g_opt"],          dtype=float)
                th    = np.array(nested_res["test_best_threshold"],   dtype=float)
                
                roc   = np.array(nested_res["test_roc_auc"],         dtype=float)
                pr    = np.array(nested_res["test_pr_auc"],          dtype=float)
                lift  = np.array(nested_res["test_lift"],            dtype=float)
                auprg = np.array(nested_res["test_auprg"],           dtype=float)
                f1g_e = np.array(nested_res["test_expected_f1g"],    dtype=float)

                n_valid_folds = int((~np.isnan(bal50)).sum())

                # Almacenar promedios
                res_emb["pooling_name"].append(pooling_name)
                res_emb["cv_roc_auc"].append(np.nanmean(roc))
                res_emb["cv_pr_auc"].append(np.nanmean(pr))
                res_emb["cv_lift"].append(np.nanmean(lift))
                res_emb["cv_auprg"].append(np.nanmean(auprg))
                res_emb["cv_expected_f1g"].append(np.nanmean(f1g_e))
                
                res_emb["cv_bal_accuracy_50"].append(np.nanmean(bal50))
                res_emb["cv_mcc_50"].append(np.nanmean(mcc50))
                res_emb["cv_precision_50"].append(np.nanmean(pr50))
                res_emb["cv_recall_50"].append(np.nanmean(roc50))
                res_emb["cv_pr_gain_50"].append(np.nanmean(prg50))
                res_emb["cv_f1g_50"].append(np.nanmean(f50))
                
                res_emb["cv_bal_accuracy_opt"].append(np.nanmean(balopt))
                res_emb["cv_mcc_opt"].append(np.nanmean(mccopt))
                res_emb["cv_precision_opt"].append(np.nanmean(propt))
                res_emb["cv_recall_opt"].append(np.nanmean(rocopt))
                res_emb["cv_pr_gain_opt"].append(np.nanmean(prgopt))
                res_emb["cv_f1g_opt"].append(np.nanmean(fopt))
                res_emb["cv_best_threshold"].append(np.nanmean(th))
                
                res_emb["n_valid_folds"].append(n_valid_folds)
                res_emb["opt_time"].append(elapsed)
                res_emb["estimators"].append(nested_res["estimator"])
                res_emb["fold_details"].append(nested_res["fold_details"])

                # --- COMPARATIVA DETALLADA IMPRESA EN TIEMPO REAL ---
                print(f"    Folds válidos: {n_valid_folds} | Tiempo: {elapsed:.1f}s")
                print(f"    ROC AUC: {np.nanmean(roc):.4f} | PR AUC: {np.nanmean(pr):.4f} | Lift: {np.nanmean(lift):.2f}x")
                print(f"    AUPRG (Área PR-Gain): {np.nanmean(auprg):.4f} | E[FG1] (F1-Gain Esperado): {np.nanmean(f1g_e):.4f}")
                print(f"    [PUNTOS OPERATIVOS CONCRETOS]")
                print(f"      -> Umbral Fijo [0.50]:")
                print(f"         Prec: {np.nanmean(pr50):.4f} | Rec: {np.nanmean(roc50):.4f} | Bal.Acc: {np.nanmean(bal50):.4f} | PR-Gain: {np.nanmean(prg50):.4f} | F1-Gain: {np.nanmean(f50):.4f}")
                print(f"      -> Umbral Optimizado [Promedio: {np.nanmean(th):.3f}]:")
                print(f"         Prec: {np.nanmean(propt):.4f} | Rec: {np.nanmean(rocopt):.4f} | Bal.Acc: {np.nanmean(balopt):.4f} | PR-Gain: {np.nanmean(prgopt):.4f} | F1-Gain: {np.nanmean(fopt):.4f}")
                print(f"    " + "-"*58)
                
            all_results[scenario][emb_name] = res_emb

    return all_results


### 5. Prueba de aleatoriedad

Verifica que el pipeline no aprende de artefactos estructurales: se asignan
etiquetas aleatorias y se espera accuracy ≈ 0.50. Se usa el escenario C1
(StratifiedKFold) para rapidez.

In [18]:
def sanity_check_random_labels(X, y, n_repeats=3):
    """
    Entrena con etiquetas aleatorias para confirmar ausencia de leakage estructural.
    Usa StratifiedKFold (C1) como escenario de comprobación rápida.

    Si el accuracy > 0.55, hay leakage en la representación o en el pipeline.
    """
    outer_c1 = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    print("\n=== Sanity Check: Labels Aleatorias (escenario C1) ===")
    accs = []
    for i in range(n_repeats):
        y_rand = np.random.permutation(y)
        outer_c1 = StratifiedKFold(n_splits=5, shuffle=True, random_state=i)
        res = hpm_search_nested_cv(
            X, 
            y_rand, 
            outer_c1,
            df_labeled["Effector_Group"].values,
            df_labeled["Protein_Group"].values,
            scenario,
            inner_cv = 5,
        )
        acc = np.nanmean(res["test_bal_accuracy_50"])
        accs.append(acc)
        print(f"  Repetición {i+1}: balanced accuracy = {acc:.4f}")
    print(f"  Media: {np.mean(accs):.4f}  (esperado ≈ 0.50)")
    print("=" * 45)

### 6. Tabla de resultados por escenario

In [19]:
def optresults2table_nested(all_results):
    """
    Convierte el dict de resultados (todos los escenarios) en un DataFrame.

    Cada fila corresponde a una combinación (escenario × embedding × pooling).
    Se extrae el hiperparámetro más frecuente entre los folds externos válidos.

    :param all_results: Dict devuelto por test_pooling_transforms_all_scenarios.
    :returns: pd.DataFrame ordenado por (escenario, pr_auc desc).
    """
    import pandas as pd
    def get_mode(lst): return max(set(lst), key=lst.count) if lst else np.nan
    
    rows = []
    for scenario, scenario_res in all_results.items():
        for emb_name, res in scenario_res.items():
            for i in range(len(res["pooling_name"])):

                estimators = [
                    e for e in res["estimators"][i]
                    if hasattr(e, "best_params_")
                ]

                if estimators:
                    param_keys = list(estimators[0].best_params_.keys())
                    hls_k   = next((k for k in param_keys if k.endswith("__hidden_layer_sizes")), None)
                    alpha_k = next((k for k in param_keys if k.endswith("__alpha")), None)
                    lr_k    = next((k for k in param_keys if k.endswith("__learning_rate_init")), None)
                    hls_list   = [str(e.best_params_[hls_k])   for e in estimators if hls_k]
                    alpha_list = [e.best_params_[alpha_k]       for e in estimators if alpha_k]
                    lr_list    = [e.best_params_[lr_k]          for e in estimators if lr_k]
                    ...
                else:
                    hls_list = alpha_list = lr_list = []

                rows.append({
                    "scenario":       scenario,
                    "representation": emb_name,
                    "pooling":        res["pooling_name"][i],
                    "roc_auc":          res["cv_roc_auc"][i],
                    "pr_auc":           res["cv_pr_auc"][i],
                    "pr_auc_lift":      res["cv_lift"][i],
                    "auprg":            res["cv_auprg"][i],
                    "expected_f1_gain": res["cv_expected_f1g"][i],
                    
                    # Umbral 50
                    "precision_50":     res["cv_precision_50"][i],
                    "recall_50":        res["cv_recall_50"][i],
                    "bal_accuracy_50":  res["cv_bal_accuracy_50"][i],
                    "mcc_50":           res["cv_mcc_50"][i],
                    "pr_gain_50":       res["cv_pr_gain_50"][i],
                    "f1_gain_50":       res["cv_f1g_50"][i],
                    
                    # Umbral Optimizado
                    "best_threshold":   res["cv_best_threshold"][i],
                    "precision_opt":    res["cv_precision_opt"][i],
                    "recall_opt":       res["cv_recall_opt"][i],
                    "bal_accuracy_opt": res["cv_bal_accuracy_opt"][i],
                    "mcc_opt":          res["cv_mcc_opt"][i],
                    "pr_gain_opt":      res["cv_pr_gain_opt"][i],
                    "f1_gain_opt":      res["cv_f1g_opt"][i],
                    
                    "time_s":           res["opt_time"][i],
                    "best_hidden_layer_sizes": get_mode(hls_list),
                    "best_alpha":              get_mode(alpha_list),
                    "best_lr_init":            get_mode(lr_list),
                    "n_valid_folds":  res["n_valid_folds"][i],
                })

    df = pd.DataFrame(rows)
    return df.sort_values(["scenario", "pr_auc"], ascending=[True, False]).reset_index(drop=True)

### 6b. Tabla de resultados por fold individual

Muestra qué grupo o combinación estaba en el test set de cada fold, su composición (n_pos, n_neg) y las métricas obtenidas. Especialmente útil cuando hay pocos folds (C3 con 4 folds) y la varianza entre folds es informativa por sí misma.

In [20]:
def fold_detail_table(all_results: dict) -> pd.DataFrame:
    """
    Genera una tabla con los resultados detallados por fold individual.

    Para cada combinación (escenario × embedding × pooling × fold) muestra:
      - fold_id  : grupo o combinación dejada en test (e.g. "EG1__PG3" en C3)
      - n_test   : número total de muestras en el test set de ese fold
      - n_test_pos / n_test_neg : composición del test set
      - accuracy, roc_auc, pr_auc : métricas de ese fold concreto
      - best_C, best_penalty : hiperparámetros seleccionados en el inner CV
      - note     : 'ok' | 'una clase' | 'vacío' | 'error'

    Útil especialmente en C3 con pocos folds: permite ver qué combinación
    biológica produce mejor o peor rendimiento y por qué (tamaño, desbalance).

    :param all_results: Dict devuelto por test_pooling_transforms_all_scenarios.
    :returns: pd.DataFrame ordenado por (scenario, representation, pooling, fold_id).
    """
    import pandas as pd
    
    rows = []
    for scenario, scenario_res in all_results.items():
        for emb_name, res in scenario_res.items():
            for i, pooling_name in enumerate(res["pooling_name"]):
                for fold_info in res["fold_details"][i]:
                    rows.append({
                        "scenario":       scenario,
                        "representation": emb_name,
                        "pooling":        pooling_name,
                        "fold_id":        fold_info["fold_id"],
                        "note":           fold_info["note"],
                        "n_test":         fold_info.get("n_test", 0),
                        "n_test_pos":     fold_info.get("n_test_pos", 0),
                        "roc_auc":        fold_info.get("roc_auc", np.nan),
                        "pr_auc":         fold_info.get("pr_auc", np.nan),
                        "lift":           fold_info.get("lift", np.nan),
                        "auprg":          fold_info.get("auprg", np.nan),
                        "expected_f1g":   fold_info.get("expected_f1g", np.nan),
                        
                        # Detalles 50
                        "precision_50":    fold_info.get("precision_50", np.nan),
                        "recall_50":       fold_info.get("recall_50", np.nan),
                        "bal_accuracy_50": fold_info.get("bal_accuracy_50", np.nan),
                        "mcc_50":          fold_info.get("mcc_50", np.nan),
                        "pr_gain_50":      fold_info.get("pr_gain_50", np.nan),
                        "f1g_50":          fold_info.get("f1g_50", np.nan),
                        
                        # Detalles Opt
                        "best_threshold":  fold_info.get("best_threshold", np.nan),
                        "precision_opt":   fold_info.get("precision_opt", np.nan),
                        "recall_opt":      fold_info.get("recall_opt", np.nan),
                        "bal_accuracy_opt": fold_info.get("bal_accuracy_opt", np.nan),
                        "mcc_opt":          fold_info.get("mcc_opt", np.nan),
                        "pr_gain_opt":      fold_info.get("pr_gain_opt", np.nan),
                        "f1g_opt":          fold_info.get("f1g_opt", np.nan),
                        
                        "best_hidden_layer_sizes": fold_info["best_hidden_layer_sizes"],
                        "best_alpha":              fold_info["best_alpha"],
                        "best_lr_init":            fold_info["best_lr_init"],
                    })
                    
    df = pd.DataFrame(rows)
    return df.sort_values(
        ["scenario", "representation", "pooling", "fold_id"]
    ).reset_index(drop=True)

### 7. Modelo final

Una vez evaluado el rendimiento por escenario, entrenamos un modelo único con
**todos los datos etiquetados** usando la mejor configuración. Este es el modelo
que se desplegará en inferencia.

Separar evaluación (nested CV) de entrenamiento final es el flujo estándar:
el nested CV estima el rendimiento real; el modelo final maximiza la información
disponible (Cawley & Talbot, 2010, *JMLR*).

In [21]:
def train_final_model(X, y, best_params):
    """
    Entrena el modelo final MLP sobre todos los datos etiquetados.
    Los pesos muestrales balanceados compensan el desbalance de clases,
    ya que MLPClassifier no soporta class_weight nativamente.
    """
    sample_weight = compute_sample_weight('balanced', y)
    final_model = make_pipeline(
        StandardScaler(),
        MLPClassifier(
            hidden_layer_sizes=eval(best_params["hidden_layer_sizes"])
                if isinstance(best_params["hidden_layer_sizes"], str)
                else best_params["hidden_layer_sizes"],
            alpha=best_params["alpha"],
            learning_rate_init=best_params["learning_rate_init"],
            activation='relu',
            solver='adam',
            max_iter=500,
            early_stopping=False,
            random_state=42,
        )
    )
    final_model.fit(X, y, mlpclassifier__sample_weight=sample_weight)
    print(f"Modelo final entrenado sobre {len(y)} instancias con params: {best_params}")
    return final_model

### 8. Inferencia sobre datos dudosos y clasificación del nivel de confianza

Tras entrenar el modelo final, se aplica a los pares dudosos (zona Q2-Q3 del ranking
topológico). Para cada pareja dudosa se asigna un **nivel de confianza** de la
predicción según si el modelo ha visto durante el entrenamiento:

- `C1` — mismo grupo efector **y** mismo grupo proteína (máxima confianza).
- `C2E` — mismo grupo efector, grupo proteína nuevo.
- `C2P` — grupo efector nuevo, mismo grupo proteína.
- `C3` — ambos grupos nuevos (mínima confianza; extrapolación total).

Esto sigue directamente el esquema de la Figura 3 del documento de selección de negativos.

In [22]:
# def classify_inference_confidence(
#     unknown_meta_df: pd.DataFrame,
#     train_meta_df: pd.DataFrame,
#     eg_col: str = "Effector_Group",
#     pg_col: str = "Protein_Group",
# ) -> pd.Series:
#     """
#     Asigna a cada pareja dudosa su nivel de confianza de predicción (C1/C2E/C2P/C3)
#     en función de si el modelo vio durante entrenamiento el grupo de efector y/o proteína.

#     :param unknown_meta_df:  DataFrame con los pares dudosos (muestra × metadatos).
#     :param train_meta_df:    DataFrame con los pares de entrenamiento.
#     :param eg_col:           Nombre de columna de grupo de efector.
#     :param pg_col:           Nombre de columna de grupo de proteína.
#     :returns: pd.Series con valores 'C1', 'C2E', 'C2P', 'C3' para cada pareja dudosa.
#     """
#     seen_eg = set(train_meta_df[eg_col])
#     seen_pg = set(train_meta_df[pg_col])

#     def _level(row):
#         has_eg = row[eg_col] in seen_eg
#         has_pg = row[pg_col] in seen_pg
#         if has_eg and has_pg:
#             return "C1"
#         elif has_eg and not has_pg:
#             return "C2E"
#         elif not has_eg and has_pg:
#             return "C2P"
#         else:
#             return "C3"

#     return unknown_meta_df.apply(_level, axis=1)


# def inference_to_csv(
#     final_model,
#     X_inference,
#     sample_names: list,
#     unknown_meta_df: pd.DataFrame,
#     train_meta_df: pd.DataFrame,
#     output_path,
# ):
#     """
#     Genera predicciones sobre muestras dudosas y guarda el resultado en CSV.

#     Columnas del CSV:
#       sample, effector_group, protein_group, proba_interact,
#       predicted_label, confidence_level

#     :param final_model:       Pipeline entrenado sobre todos los datos etiquetados.
#     :param X_inference:       Array de embeddings de los dudosos (n_samples, n_features).
#     :param sample_names:      Lista de nombres de muestra (alineada con X_inference).
#     :param unknown_meta_df:   DataFrame con metadatos de los dudosos.
#     :param train_meta_df:     DataFrame con metadatos de entrenamiento.
#     :param output_path:       Path del CSV de salida.
#     :returns: pd.DataFrame con las predicciones ordenadas.
#     """
#     proba = final_model.predict_proba(X_inference)[:, 1]

#     # Alinear metadatos con el orden de X_inference
#     meta_aligned = unknown_meta_df.set_index("Sample_name").loc[sample_names].reset_index()
#     meta_aligned = meta_aligned.rename(columns={"Sample_name": "sample"})

#     confidence = classify_inference_confidence(meta_aligned, train_meta_df)

#     df = meta_aligned.copy()
#     df["proba_interact"]  = proba
#     df["predicted_label"] = (proba >= 0.5).astype(int)
#     df["confidence_level"] = confidence.values

#     df = df.sort_values("proba_interact", ascending=False).reset_index(drop=True)

#     output_path = Path(output_path)
#     output_path.parent.mkdir(parents=True, exist_ok=True)
#     df.to_csv(output_path, index=False)
#     print(f"Predicciones guardadas en: {output_path}  ({len(df)} muestras)")
#     print("\nDistribución de niveles de confianza:")
#     print(df["confidence_level"].value_counts().to_string())
#     return df

In [23]:
def classify_inference_confidence(
    unknown_meta_df: pd.DataFrame,
    train_meta_df: pd.DataFrame,
    eg_col: str = "Effector_Group",
    pg_col: str = "Protein_Group",
    e_id_col: str = "Effector",   # <--- Nueva columna para el ID individual del efector
    p_id_col: str = "Protein",    # <--- Nueva columna para el ID individual de la proteína
) -> pd.DataFrame:                # <--- Ahora devuelve un DataFrame con ambas series
    """
    Asigna a cada pareja dudosa su nivel de confianza de predicción (C1/C2E/C2P/C3)
    tanto a nivel de GRUPO como a nivel INDIVIDUAL.

    :returns: pd.DataFrame con dos columnas: 'confidence_level' y 'confidence_individual_sublevel'
    """
    # Sets para nivel de GRUPO
    seen_eg = set(train_meta_df[eg_col].dropna())
    seen_pg = set(train_meta_df[pg_col].dropna())
    
    # Sets para nivel INDIVIDUAL
    seen_e_id = set(train_meta_df[e_id_col].dropna())
    seen_p_id = set(train_meta_df[p_id_col].dropna())

    # Lógica de grupos
    def _level_group(row):
        has_eg = row[eg_col] in seen_eg
        has_pg = row[pg_col] in seen_pg
        if has_eg and has_pg:   return "C1"
        if has_eg:              return "C2E"
        if has_pg:              return "C2P"
        return "C3"

    # Lógica individual (Invertida según tu especificación: C2E si vio proteína, C2P si vio efector)
    def _level_individual(row):
        has_e = row[e_id_col] in seen_e_id
        has_p = row[p_id_col] in seen_p_id
        if has_e and has_p:     return "C1"
        if not has_e and has_p: return "C2E"  # Vio proteína pero no efector
        if has_e and not has_p: return "C2P"  # Vio efector pero no proteína
        return "C3"                           # No vio ninguno

    # Construimos el DataFrame de salida
    res_df = pd.DataFrame(index=unknown_meta_df.index)
    res_df["confidence_level"] = unknown_meta_df.apply(_level_group, axis=1)
    res_df["confidence_individual_sublevel"] = unknown_meta_df.apply(_level_individual, axis=1)
    
    return res_df


def inference_to_csv(
    final_model,
    X_inference,
    sample_names: list,
    unknown_meta_df: pd.DataFrame,
    train_meta_df: pd.DataFrame,
    output_path,
    eg_col: str = "Effector_Group",
    pg_col: str = "Protein_Group",
    e_id_col: str = "Effector",   # <--- Añadido aquí también
    p_id_col: str = "Protein",    # <--- Añadido aquí también
):
    """
    Genera predicciones sobre muestras dudosas y guarda el resultado en CSV
    incluyendo subniveles de confianza individuales.
    """
    proba = final_model.predict_proba(X_inference)[:, 1]

    # Alinear metadatos con el orden de X_inference
    meta_aligned = unknown_meta_df.set_index("Sample_name").loc[sample_names].reset_index()
    meta_aligned = meta_aligned.rename(columns={"Sample_name": "sample"})

    # Calculamos ambos niveles de confianza a la vez
    confidence_df = classify_inference_confidence(
        meta_aligned, 
        train_meta_df, 
        eg_col=eg_col, 
        pg_col=pg_col, 
        e_id_col=e_id_col, 
        p_id_col=p_id_col
    )

    df = meta_aligned.copy()
    df["proba_interact"] = proba
    df["predicted_label"] = (proba >= 0.5).astype(int)
    
    # Asignamos las dos columnas resultantes
    df["confidence_level"] = confidence_df["confidence_level"].values
    df["confidence_individual_sublevel"] = confidence_df["confidence_individual_sublevel"].values

    df = df.sort_values("proba_interact", ascending=False).reset_index(drop=True)

    output_path = Path(output_path)
    output_path.parent.mkdir(parents=True, exist_ok=True)
    df.to_csv(output_path, index=False)
    
    print(f"Predicciones guardadas en: {output_path} ({len(df)} muestras)")
    print("\nDistribución de niveles de confianza (GRUPO):")
    print(df["confidence_level"].value_counts().to_string())
    print("\nDistribución de subniveles de confianza (INDIVIDUAL):")
    print(df["confidence_individual_sublevel"].value_counts().to_string())
    
    return df

# Ejecución principal

In [23]:
# ── 0. Cargar metadatos y generar splits Cx ──────────────────────────────────
# metadata.csv debe tener columnas: sample_name, effector, protein,
#                                   effector_group, protein_group, label
#meta_df = pd.read_csv(PATH_METADATA, dtype={"label": int})

# Solo pares con ambas clases en su celda (verdes + naranjas del heatmap P2)
# — los rojos y grises no participan en el entrenamiento ni en los splits
#labeled_meta = meta_df[meta_df["label"].isin([0, 1])].copy()
labeled_meta = df_labeled

# Generar (o regenerar) los splits y guardarlos
splits = build_and_save_splits(
    labeled_meta,
    output_dir=str(SPLITS_DIR),
    effector_group_col="Effector_Group",
    protein_group_col="Protein_Group",
    label_col="Label",
    sample_col="Sample_name",
    min_C3=3,
    min_C2=3,
)

print(f"\nDataset: {len(labeled_meta)} parejas  "
      f"({(labeled_meta.Label==1).sum()} pos, {(labeled_meta.Label==0).sum()} neg)")
print(f"Splits guardados en: {SPLITS_DIR}")


🔧 Generando splits Cx...

📋 Reporte de particiones:

  REPORTE DE PARTICIONES CV — ESCENARIOS Cx
  Umbral C3: ≥3 pos + ≥3 neg por combinación
  Umbral C2: ≥3 pos + ≥3 neg por grupo (celdas biclase)

─────────────────────────────────────────────────────────────────
  Escenario C3  (15 folds válidos)
─────────────────────────────────────────────────────────────────
  Fold [Cluster_0__Cluster_0          ]  train=481  test= 73 (pos=12, neg=61)  excl=408  ✅
  Fold [Cluster_0__Cluster_1          ]  train=594  test= 38 (pos=10, neg=28)  excl=330  ✅
  Fold [Cluster_0__Cluster_3          ]  train=435  test=109 (pos=17, neg=92)  excl=418  ✅
  Fold [Cluster_0__Cluster_4          ]  train=465  test= 77 (pos=17, neg=60)  excl=420  ✅
  Fold [Cluster_1__Cluster_3          ]  train=459  test= 78 (pos=10, neg=68)  excl=425  ✅
  Fold [Cluster_1__Cluster_4          ]  train=515  test= 72 (pos=6, neg=66)  excl=375  ✅
  Fold [Cluster_2__Cluster_4          ]  train=681  test= 16 (pos=11, neg=5)  excl=265  

  Guardado: /home/jovyan/TFG/CV_Kmeans_con_EspS_NleK_all_models/results/splits/splits_C3_roles.csv  (962 muestras × 15 folds)
  Guardado: /home/jovyan/TFG/CV_Kmeans_con_EspS_NleK_all_models/results/splits/splits_C3_meta.json
  Guardado: /home/jovyan/TFG/CV_Kmeans_con_EspS_NleK_all_models/results/splits/splits_C2E_roles.csv  (962 muestras × 7 folds)
  Guardado: /home/jovyan/TFG/CV_Kmeans_con_EspS_NleK_all_models/results/splits/splits_C2E_meta.json
  Guardado: /home/jovyan/TFG/CV_Kmeans_con_EspS_NleK_all_models/results/splits/splits_C2P_roles.csv  (962 muestras × 5 folds)
  Guardado: /home/jovyan/TFG/CV_Kmeans_con_EspS_NleK_all_models/results/splits/splits_C2P_meta.json
  Guardado: /home/jovyan/TFG/CV_Kmeans_con_EspS_NleK_all_models/results/splits/splits_report.txt

Dataset: 962 parejas  (199 pos, 763 neg)
Splits guardados en: /home/jovyan/TFG/CV_Kmeans_con_EspS_NleK_all_models/results/splits


In [24]:
# ── 1. Definir transformaciones de pooling a comparar ────────────────────────
test_transforms = {
    "single_embeddings": [
        lambda x: np.mean(x, axis=0),
        lambda x: np.max(x, axis=0),
        lambda x: np.min(x, axis=0),
    ],
    "pair_embeddings": [
        lambda x: np.mean(x, axis=(0, 1)),
        lambda x: np.max(x, axis=(0, 1)),
        lambda x: np.min(x, axis=(0, 1)),
    ],
}

warnings.filterwarnings("ignore")

In [25]:
# Diagnóstico rápido
nombres_en_disco = [d.name for d in OUTPUT_DIR.iterdir() if d.is_dir() and ".ipynb" not in d.name]
nombres_en_excel = labeled_meta["Sample_name"].unique().tolist()

print(f"Total carpetas en disco: {len(nombres_en_disco)}")
print(f"Total IDs en Excel: {len(nombres_en_excel)}")
print(f"Ejemplo disco: {nombres_en_disco[0] if nombres_en_disco else 'VACÍO'}")
print(f"Ejemplo excel: {nombres_en_excel[0] if nombres_en_excel else 'VACÍO'}")

labeled_meta

Total carpetas en disco: 5472
Total IDs en Excel: 962
Ejemplo disco: P35283_EspL
Ejemplo excel: Q62159_EspM


,Effector,Protein,Is_Connected,Effector_Group,Protein_Group,Label,Sample_name
0,EspM,Rhoc,True,Cluster_6,Cluster_3,1,Q62159_EspM
1,NleB,Ripk1,True,Cluster_0,Cluster_4,1,Q60855_NleB
2,NleB,Nfkb1,True,Cluster_0,Cluster_4,1,P25799_NleB
3,EspH,Rac2,True,Cluster_1,Cluster_3,1,Q05144_EspH
4,NleD,Mapkapk2,True,Cluster_6,Cluster_1,1,P49138_NleD
...,...,...,...,...,...,...,...
958,NleK,Rab17,False,Cluster_1,Cluster_3,0,P35292_NleK
959,NleK,Rab11b,False,Cluster_1,Cluster_3,0,P46638_NleK
960,NleK,Rab11fip3,False,Cluster_1,Cluster_0,0,Q8CHD8_NleK
961,NleK,Rhod,False,Cluster_1,Cluster_3,0,P97348_NleK


In [26]:
# ── 2. Nested CV para los cuatro escenarios ──────────────────────────────────
# Ejecutar los cuatro escenarios (puede llevar varios minutos según el tamaño del dataset)
all_results = test_pooling_transforms_all_scenarios(
    test_transforms,
    labeled_dir=OUTPUT_DIR,
    splits_dir=SPLITS_DIR,
    metadata_df=labeled_meta,
    scenarios=["C1", "C2E", "C2P", "C3"],
)


######################################################################
#  ESCENARIO C1
######################################################################

  Embedding: single_embeddings  (3 poolings)


✅ Éxito: Se cargaron 962 muestras de las 962 del Excel.
[[[ -202.8    -406.2    -306.    ...   -69.1    -330.2    -181.8  ]
  [ -296.2    -373.     -234.1   ...   -94.2    -364.8    -336.5  ]
  [ -358.2    -305.8    -165.1   ...   -31.33   -379.2    -384.   ]
  ...
  [ -374.5    -307.8    -197.1   ...   -98.2    -565.5    -397.5  ]
  [ -203.6    -248.9    -126.2   ...   -67.75   -435.2    -248.4  ]
  [ -304.8    -282.2    -203.5   ...    14.234  -462.2    -416.5  ]]

 [[  768.      338.      119.    ...   366.      366.      240.   ]
  [  760.      420.      362.    ...   346.      262.      127.5  ]
  [  510.      504.      378.    ...   520.      192.      189.   ]
  ...
  [  466.      470.      462.    ...   478.      184.       33.5  ]
  [  532.      400.      382.    ...   422.      164.      144.   ]
  [  692.      434.      414.    ...   528.      158.      264.   ]]

 [[-1056.    -1224.     -820.    ...  -540.     -836.     -752.   ]
  [-1184.    -1004.     -800.    ...  -560. 

    folds: total=5  válidos=5  NaN=0
    Folds válidos: 5 | Tiempo: 128.8s
    ROC AUC: 0.9516 | PR AUC: 0.9005 | Lift: 4.35x
    AUPRG (Área PR-Gain): 0.9786 | E[FG1] (F1-Gain Esperado): 0.7393
    [PUNTOS OPERATIVOS CONCRETOS]
      -> Umbral Fijo [0.50]:
         Prec: 0.8692 | Rec: 0.8396 | Bal.Acc: 0.9028 | PR-Gain: 0.9601 | F1-Gain: 0.9543
      -> Umbral Optimizado [Promedio: 0.479]:
         Prec: 0.9016 | Rec: 0.8397 | Bal.Acc: 0.9074 | PR-Gain: 0.9710 | F1-Gain: 0.9599
    ----------------------------------------------------------

  --- single_embeddings | max ---


    folds: total=5  válidos=5  NaN=0
    Folds válidos: 5 | Tiempo: 88.5s
    ROC AUC: 0.9403 | PR AUC: 0.8783 | Lift: 4.25x
    AUPRG (Área PR-Gain): 0.9749 | E[FG1] (F1-Gain Esperado): 0.7374
    [PUNTOS OPERATIVOS CONCRETOS]
      -> Umbral Fijo [0.50]:
         Prec: 0.8633 | Rec: 0.7241 | Bal.Acc: 0.8470 | PR-Gain: 0.9585 | F1-Gain: 0.9268
      -> Umbral Optimizado [Promedio: 0.382]:
         Prec: 0.8371 | Rec: 0.8242 | Bal.Acc: 0.8892 | PR-Gain: 0.9464 | F1-Gain: 0.9442
    ----------------------------------------------------------

  --- single_embeddings | min ---


    folds: total=5  válidos=5  NaN=0
    Folds válidos: 5 | Tiempo: 73.1s
    ROC AUC: 0.9307 | PR AUC: 0.8611 | Lift: 4.16x
    AUPRG (Área PR-Gain): 0.9696 | E[FG1] (F1-Gain Esperado): 0.7348
    [PUNTOS OPERATIVOS CONCRETOS]
      -> Umbral Fijo [0.50]:
         Prec: 0.8224 | Rec: 0.7242 | Bal.Acc: 0.8405 | PR-Gain: 0.9416 | F1-Gain: 0.9169
      -> Umbral Optimizado [Promedio: 0.349]:
         Prec: 0.8224 | Rec: 0.7791 | Bal.Acc: 0.8659 | PR-Gain: 0.9408 | F1-Gain: 0.9321
    ----------------------------------------------------------

  Embedding: pair_embeddings  (3 poolings)


...

    folds: total=15  válidos=15  NaN=0
    Folds válidos: 15 | Tiempo: 595.6s
    ROC AUC: 0.5118 | PR AUC: 0.3945 | Lift: 1.36x
    AUPRG (Área PR-Gain): 0.2164 | E[FG1] (F1-Gain Esperado): 0.3614
    [PUNTOS OPERATIVOS CONCRETOS]
      -> Umbral Fijo [0.50]:
         Prec: 0.3146 | Rec: 0.4695 | Bal.Acc: 0.5276 | PR-Gain: 0.2506 | F1-Gain: 0.3482
      -> Umbral Optimizado [Promedio: 0.215]:
         Prec: 0.3844 | Rec: 0.7989 | Bal.Acc: 0.5992 | PR-Gain: 0.3053 | F1-Gain: 0.5355
    ----------------------------------------------------------

  Embedding: pair_embeddings  (3 poolings)


✅ Éxito: Se cargaron 962 muestras de las 962 del Excel.
[[[ 5.2539e+00  9.8203e+00 -2.8789e+00 ...  4.4102e+00 -2.0430e+00
    2.9570e+00]
  [ 5.3516e+00 -1.1152e+00 -7.4414e-01 ...  4.6558e-01 -1.0557e+00
    5.0117e+00]
  [ 6.0234e+00 -4.8125e+00  7.5293e-01 ...  6.7090e-01  5.7324e-01
    5.0430e+00]
  ...
  [ 6.6055e+00 -5.2578e+00  1.3662e+00 ...  3.9648e-01 -2.3887e+00
    6.9336e+00]
  [ 2.9688e+00  1.0523e+01 -7.0850e-01 ...  1.6426e+00 -4.1719e+00
    2.9180e+00]
  [ 1.1531e+01 -1.3555e+01 -1.0850e+00 ...  5.5391e+00  9.0625e+00
    7.4258e+00]]

 [[ 8.7600e+02  8.4800e+02  8.3200e+02 ...  8.7200e+02  8.0500e+01
    2.1760e+03]
  [ 7.7200e+02  8.8000e+02  7.1200e+02 ...  8.3200e+02  1.0450e+02
    2.1280e+03]
  [ 7.4800e+02  7.9200e+02  7.5200e+02 ...  7.5600e+02  1.0050e+02
    2.0800e+03]
  ...
  [ 7.3600e+02  7.4000e+02  5.7600e+02 ...  6.8400e+02  7.9500e+01
    2.2080e+03]
  [ 8.8800e+02  8.8400e+02  7.8400e+02 ...  7.9600e+02  5.1250e+01
    2.1920e+03]
  [ 7.1600e+02  8

    folds: total=15  válidos=15  NaN=0
    Folds válidos: 15 | Tiempo: 1328.5s
    ROC AUC: 0.5295 | PR AUC: 0.4041 | Lift: 1.34x
    AUPRG (Área PR-Gain): 0.3238 | E[FG1] (F1-Gain Esperado): 0.4341
    [PUNTOS OPERATIVOS CONCRETOS]
      -> Umbral Fijo [0.50]:
         Prec: 0.2246 | Rec: 0.3442 | Bal.Acc: 0.4753 | PR-Gain: 0.1551 | F1-Gain: 0.2180
      -> Umbral Optimizado [Promedio: 0.370]:
         Prec: 0.3857 | Rec: 0.4333 | Bal.Acc: 0.5270 | PR-Gain: 0.3635 | F1-Gain: 0.3560
    ----------------------------------------------------------

  --- pair_embeddings | max ---


    folds: total=15  válidos=15  NaN=0
    Folds válidos: 15 | Tiempo: 481.9s
    ROC AUC: 0.4942 | PR AUC: 0.3714 | Lift: 1.36x
    AUPRG (Área PR-Gain): 0.1957 | E[FG1] (F1-Gain Esperado): 0.3596
    [PUNTOS OPERATIVOS CONCRETOS]
      -> Umbral Fijo [0.50]:
         Prec: 0.2893 | Rec: 0.5454 | Bal.Acc: 0.4984 | PR-Gain: 0.1611 | F1-Gain: 0.3485
      -> Umbral Optimizado [Promedio: 0.325]:
         Prec: 0.3621 | Rec: 0.6751 | Bal.Acc: 0.5600 | PR-Gain: 0.2671 | F1-Gain: 0.4748
    ----------------------------------------------------------

  --- pair_embeddings | min ---


    folds: total=15  válidos=15  NaN=0
    Folds válidos: 15 | Tiempo: 457.8s
    ROC AUC: 0.4617 | PR AUC: 0.3556 | Lift: 1.21x
    AUPRG (Área PR-Gain): 0.1512 | E[FG1] (F1-Gain Esperado): 0.3352
    [PUNTOS OPERATIVOS CONCRETOS]
      -> Umbral Fijo [0.50]:
         Prec: 0.2800 | Rec: 0.3195 | Bal.Acc: 0.4574 | PR-Gain: 0.1202 | F1-Gain: 0.1593
      -> Umbral Optimizado [Promedio: 0.067]:
         Prec: 0.3240 | Rec: 0.6436 | Bal.Acc: 0.5185 | PR-Gain: 0.1879 | F1-Gain: 0.3683
    ----------------------------------------------------------


In [27]:
# Guardamos el objeto ante cualquier imprevisto
import joblib

fichero_salida = joblib.dump(all_results, "checkpoint_CV_kmeans_PRG_con_EspS_NleK_MLP.joblib")

print(f"\n[OK] ¡Análisis de escenarios completado!")
print(f"[OK] Se han guardado todas las métricas, precisiones, recalls y el F1-Gain esperado por fold.")
print(f"[OK] Archivo binario guardado en: '{fichero_salida}'")


[OK] ¡Análisis de escenarios completado!
[OK] Se han guardado todas las métricas, precisiones, recalls y el F1-Gain esperado por fold.
[OK] Archivo binario guardado en: '['checkpoint_CV_kmeans_PRG_con_EspS_NleK_MLP.joblib']'


In [24]:
# Get the joblib back
import joblib

all_results = joblib.load("checkpoint_CV_kmeans_PRG_con_EspS_NleK_MLP.joblib")

In [25]:
# ── 3a. Tabla resumen de resultados por escenario ────────────────────────
df_results = optresults2table_nested(all_results)
display(df_results)
df_results.to_csv(OUTPUT_RESULTS / "cv_results_all_scenarios_estricto_kmeans_PRG_con_EspS_NleK_MLP.csv", index=False)
print(f"Tabla resumen guardada en: {OUTPUT_RESULTS / 'cv_results_all_scenarios_estricto_kmeans_PRG_con_EspS_NleK_MLP.csv'}")

,scenario,representation,pooling,roc_auc,pr_auc,pr_auc_lift,auprg,expected_f1_gain,precision_50,recall_50,...,recall_opt,bal_accuracy_opt,mcc_opt,pr_gain_opt,f1_gain_opt,time_s,best_hidden_layer_sizes,best_alpha,best_lr_init,n_valid_folds
0,C1,single_embeddings,mean,0.951613,0.900538,4.354138,0.978573,0.739286,0.869206,0.839615,...,0.839744,0.907423,0.836818,0.971021,0.959896,128.827395,"(256, 128, 64)",0.0100,0.0001,5
1,C1,single_embeddings,max,0.940323,0.878342,4.246921,0.974889,0.737445,0.863329,0.724103,...,0.824231,0.889184,0.783569,0.946396,0.944228,88.510778,"(512, 256, 128)",0.0100,0.0010,5
2,C1,single_embeddings,min,0.930692,0.861132,4.164396,0.969623,0.734811,0.822366,0.724231,...,0.779103,0.865923,0.747442,0.940808,0.932089,73.121944,"(512, 256, 128)",0.0100,0.0010,5
3,C1,pair_embeddings,min,0.926243,0.833192,4.028760,0.961016,0.730508,0.815693,0.728718,...,0.753590,0.854474,0.736048,0.942191,0.927429,106.066480,"(512, 256, 128)",0.0100,0.0010,5
4,C1,pair_embeddings,max,0.911851,0.822998,3.980506,0.955018,0.727509,0.785386,0.703974,...,0.748974,0.846301,0.705865,0.926618,0.915435,118.229836,"(512, 256, 128)",0.0100,0.0010,5
5,C1,pair_embeddings,mean,0.893552,0.793884,3.838251,0.944214,0.722107,0.752345,0.698333,...,0.738590,0.837815,0.684611,0.915453,0.911123,286.115262,"(512, 256, 128)",0.0010,0.0001,5
6,C2E,single_embeddings,mean,0.743450,0.557236,2.293851,0.562707,0.542818,0.415650,0.356497,...,0.636823,0.711032,0.405255,0.658076,0.673979,195.645176,"(128, 64, 32)",0.0100,0.0001,7
7,C2E,single_embeddings,max,0.658624,0.440282,1.576880,0.364660,0.472392,0.272039,0.185384,...,0.417036,0.621962,0.224236,0.432082,0.511639,186.491986,"(512, 256, 128)",0.0010,0.0001,7
8,C2E,pair_embeddings,min,0.587080,0.430404,1.826066,0.349415,0.425151,0.311679,0.222305,...,0.482420,0.637159,0.259537,0.593465,0.458166,103.224508,"(128, 64, 32)",0.0100,0.0010,7
9,C2E,pair_embeddings,max,0.622546,0.413369,1.598499,0.352541,0.445606,0.304130,0.379628,...,0.538622,0.652044,0.272291,0.545061,0.563942,121.797071,"(256, 128, 64)",0.0010,0.0001,7


Tabla resumen guardada en: /home/jovyan/TFG/CV_Kmeans_con_EspS_NleK_all_models/results/cv_results_all_scenarios_estricto_kmeans_PRG_con_EspS_NleK_MLP.csv


In [26]:
# ── 3b. Tabla detallada por fold individual ───────────────────────────────
# Muestra para cada fold: grupo en test, composición (n_pos/n_neg) y métricas.
# Especialmente útil en C3 con pocos folds: la varianza entre folds es
# información real sobre robustez que hay que reportar (Chicco & Jurman, 2020).
df_folds = fold_detail_table(all_results)
#df_folds = pd.read_csv(OUTPUT_RESULTS / "cv_results_per_fold_g13sep_estricto.csv")

# Imprimir la mejor combinación embedding×pooling por escenario
import numpy as np

# Definimos las columnas que queremos inspeccionar al detalle por cada fold
# Incluimos la comparativa de umbrales (50 vs opt), precisión, recall, lift, auprg y e[fg1]
cols = [
    "fold_id", "note", "n_test", "n_test_pos",
    "roc_auc", "pr_auc", "lift", "auprg", "expected_f1g",
    "best_threshold", "precision_50", "recall_50", "f1g_50",
    "precision_opt", "recall_opt", "f1g_opt", "best_hidden_layer_sizes", "best_alpha", "best_lr_init"
]

for scenario in ["C3", "C2E", "C2P", "C1"]:
    if scenario not in df_results["scenario"].values:
        continue
        
    # === MODIFICACIÓN 1: Ordenamos por 'auprg' (métricamente más robusto que pr_auc) ===
    best_combo = (
        df_results[df_results["scenario"] == scenario]
        .sort_values("auprg", ascending=False)
        .iloc[0]
    )

    # Filtrar el detalle de folds para el mejor modelo de este escenario
    sub = df_folds[
        (df_folds["scenario"]       == scenario) &
        (df_folds["representation"] == best_combo["representation"]) &
        (df_folds["pooling"]        == best_combo["pooling"])
    ]

    # === MODIFICACIÓN 2: Calcular el Error Estándar (SEM) sobre el AUPRG ===
    n_folds = len(sub)
    # Usamos ddof=1 para la desviación estándar muestral unbiased
    auprg_std = sub['auprg'].std(ddof=1) / np.sqrt(n_folds) if n_folds > 1 else 0.0

    print(f"\n{'='*85}")
    print(f"  {scenario} — {best_combo['representation']} | {best_combo['pooling']}")
    print(f"  (Media AUPRG = {best_combo['auprg']:.4f} ± {auprg_std:.4f} | E[FG1] = {best_combo['expected_f1_gain']:.4f})")
    print(f"{'='*85}")
    
    # Asegurar que solo imprimimos las columnas que realmente existen en el DataFrame
    existing_cols = [c for c in cols if c in sub.columns]
    display(sub[existing_cols].reset_index(drop=True))

# Guardar el archivo final en el directorio de salida correspondiente
output_path = OUTPUT_RESULTS / "cv_results_per_fold_estricto_kmeans_PRG_con_EspS_NleK_MLP.csv"
df_folds.to_csv(output_path, index=False)
print(f"\nTabla completa por fold guardada en: {output_path}")


  C3 — single_embeddings | max
  (Media AUPRG = 0.4060 ± 0.0766 | E[FG1] = 0.4646)


,fold_id,note,n_test,n_test_pos,roc_auc,pr_auc,lift,auprg,expected_f1g,best_threshold,precision_50,recall_50,f1g_50,precision_opt,recall_opt,f1g_opt,best_hidden_layer_sizes,best_alpha,best_lr_init
0,Cluster_0__Cluster_0,ok,73,12,0.7090,0.3070,1.8677,0.6013,0.6098,0.5643,0.2727,0.7500,0.7049,0.2812,0.7500,0.7158,"(256, 128, 64)",0.0001,0.0010
1,Cluster_0__Cluster_1,ok,38,10,0.7536,0.6517,2.4765,0.7475,0.6237,0.9702,0.3529,0.6000,0.5536,0.7143,0.5000,0.7500,"(256, 128, 64)",0.0001,0.0010
2,Cluster_0__Cluster_3,ok,109,17,0.9079,0.6269,4.0195,0.8956,0.7010,0.9900,0.1604,1.0000,0.5163,0.2581,0.9412,0.7286,"(256, 128, 64)",0.0100,0.0001
3,Cluster_0__Cluster_4,ok,77,17,0.3225,0.1656,0.7499,0.0008,0.2505,0.1684,0.2154,0.8235,0.4536,0.2329,1.0000,0.5333,"(512, 256, 128)",0.0010,0.0001
4,Cluster_1__Cluster_3,ok,78,10,0.6574,0.1794,1.3992,0.1382,0.3293,0.0694,0.0000,0.0000,0.0000,0.2400,0.6000,0.7181,"(128, 64, 32)",0.0001,0.0001
5,Cluster_1__Cluster_4,ok,72,6,0.7096,0.1526,1.8310,0.4139,0.4570,0.7821,0.1471,0.8333,0.7273,0.1786,0.8333,0.7818,"(512, 256, 128)",0.0010,0.0010
6,Cluster_2__Cluster_4,ok,16,11,0.0000,0.5012,0.7290,0.0875,0.2938,0.0100,0.1667,0.0909,0.0000,0.4444,0.3636,0.0000,"(256, 128, 64)",0.0001,0.0010
7,Cluster_3__Cluster_4,ok,31,6,0.6933,0.2992,1.5459,0.2360,0.3963,0.3763,0.2941,0.8333,0.6880,0.3158,1.0000,0.7400,"(512, 256, 128)",0.0001,0.0001
8,Cluster_4__Cluster_0,ok,10,5,0.6000,0.5926,1.1852,0.3444,0.4222,0.0100,0.0000,0.0000,0.0000,0.6250,1.0000,0.7000,"(256, 128, 64)",0.0001,0.0010
9,Cluster_4__Cluster_3,ok,17,7,0.5286,0.4486,1.0894,0.1044,0.3022,0.2773,0.4615,0.8571,0.5333,0.5000,1.0000,0.6500,"(128, 64, 32)",0.0010,0.0001



  C2E — single_embeddings | mean
  (Media AUPRG = 0.5627 ± 0.1273 | E[FG1] = 0.5428)


,fold_id,note,n_test,n_test_pos,roc_auc,pr_auc,lift,auprg,expected_f1g,best_threshold,precision_50,recall_50,f1g_50,precision_opt,recall_opt,f1g_opt,best_hidden_layer_sizes,best_alpha,best_lr_init
0,Cluster_0,ok,305,58,0.8302,0.5104,2.6839,0.8171,0.6646,0.3763,0.4815,0.6724,0.8164,0.4674,0.7414,0.8253,"(128, 64, 32)",0.0100,0.0001
1,Cluster_1,ok,250,19,0.7250,0.1430,1.8813,0.3197,0.4230,0.0100,0.1333,0.1053,0.3831,0.1429,0.4211,0.6967,"(512, 256, 128)",0.0001,0.0010
2,Cluster_2,ok,28,21,0.4558,0.7673,1.0230,0.0000,0.2500,0.0100,0.0000,0.0000,0.0000,0.8182,0.4286,0.0000,"(128, 64, 32)",0.0100,0.0001
3,Cluster_3,ok,113,11,0.7772,0.3968,4.0766,0.8635,0.6818,0.3070,0.4000,0.3636,0.8248,0.4118,0.6364,0.8922,"(256, 128, 64)",0.0010,0.0001
4,Cluster_4,ok,37,16,0.7768,0.7968,1.8427,0.5638,0.5319,0.0397,0.5000,0.6875,0.4459,0.5714,1.0000,0.7143,"(256, 128, 64)",0.0010,0.0010
5,Cluster_5,ok,29,6,0.6884,0.4049,1.9568,0.4444,0.5301,0.1684,0.5000,0.1667,0.2174,0.6667,0.3333,0.6739,"(128, 64, 32)",0.0010,0.0001
6,Cluster_6,ok,200,68,0.9508,0.8815,2.5926,0.9305,0.7183,0.0199,0.8947,0.5000,0.7121,0.8243,0.8971,0.9155,"(512, 256, 128)",0.0100,0.0001



  C2P — pair_embeddings | min
  (Media AUPRG = 0.7249 ± 0.0645 | E[FG1] = 0.6148)


,fold_id,note,n_test,n_test_pos,roc_auc,pr_auc,lift,auprg,expected_f1g,best_threshold,precision_50,recall_50,f1g_50,precision_opt,recall_opt,f1g_opt,best_hidden_layer_sizes,best_alpha,best_lr_init
0,Cluster_0,ok,249,36,0.8251,0.5189,3.5890,0.8808,0.6914,0.6336,0.4884,0.5833,0.8511,0.5000,0.5833,0.8551,"(512, 256, 128)",0.0100,0.0010
1,Cluster_1,ok,101,27,0.8498,0.6877,2.5726,0.7971,0.6535,0.0199,0.6400,0.5926,0.7720,0.6364,0.7778,0.8436,"(128, 64, 32)",0.0001,0.0010
2,Cluster_2,ok,12,3,0.7037,0.6111,2.4444,0.7176,0.6088,0.0100,0.3333,0.3333,0.3333,0.5000,0.6667,0.7500,"(512, 256, 128)",0.0001,0.0010
3,Cluster_3,ok,331,79,0.7021,0.4225,1.7703,0.4933,0.4980,0.0298,0.3818,0.2658,0.3133,0.4082,0.5063,0.6199,"(512, 256, 128)",0.0100,0.0001
4,Cluster_4,ok,269,54,0.7148,0.4960,2.4706,0.7357,0.6223,0.9900,0.2952,0.5741,0.6071,0.5556,0.4630,0.7539,"(512, 256, 128)",0.0001,0.0010



  C1 — single_embeddings | mean
  (Media AUPRG = 0.9786 ± 0.0023 | E[FG1] = 0.7393)


,fold_id,note,n_test,n_test_pos,roc_auc,pr_auc,lift,auprg,expected_f1g,best_threshold,precision_50,recall_50,f1g_50,precision_opt,recall_opt,f1g_opt,best_hidden_layer_sizes,best_alpha,best_lr_init
0,fold_0,ok,193,40,0.9405,0.9024,4.3541,0.9798,0.7399,0.5940,0.9143,0.8000,0.9551,0.9697,0.8000,0.9632,"(256, 128, 64)",0.0001,0.0001
1,fold_1,ok,193,40,0.9474,0.8936,4.3115,0.9793,0.7396,0.4555,0.8857,0.7750,0.9452,0.8889,0.8000,0.9510,"(256, 128, 64)",0.0010,0.0001
2,fold_2,ok,192,40,0.9437,0.8923,4.2830,0.9774,0.7387,0.3070,0.8889,0.8000,0.9507,0.8919,0.8250,0.9561,"(256, 128, 64)",0.0100,0.0010
3,fold_3,ok,192,40,0.9658,0.8973,4.3070,0.9711,0.7356,0.9009,0.8000,0.9000,0.9525,0.9167,0.8250,0.9601,"(256, 128, 64)",0.0100,0.0001
4,fold_4,ok,192,39,0.9606,0.9171,4.5151,0.9853,0.7426,0.1387,0.8571,0.9231,0.9681,0.8409,0.9487,0.9690,"(256, 128, 64)",0.0100,0.0010



Tabla completa por fold guardada en: /home/jovyan/TFG/CV_Kmeans_con_EspS_NleK_all_models/results/cv_results_per_fold_estricto_kmeans_PRG_con_EspS_NleK_MLP.csv


In [ ]:
# Volvemos a cargar los resultados para no tener que ejecutar de nuevo todo el análisis
#df_results = pd.read_csv(OUTPUT_RESULTS / "cv_results_all_scenarios_g13sep_estricto.csv")

In [27]:
# ── 4. Elegir mejor configuración (por PR AUC en C3 — escenario más exigente) ─
# Puede cambiarse a "C1" para el baseline o a otro escenario según criterio biológico
TARGET_SCENARIO = "C3"

# === MODIFICACIÓN 1: Ordenamos por 'auprg' (Área de PR-Gain) ===
best_row = (
    df_results[df_results["scenario"] == TARGET_SCENARIO]
    .sort_values("auprg", ascending=False)
    .iloc[0]
)

BEST_EMB_TYPE    = best_row["representation"]
BEST_POOLING_IDX = ["mean", "max", "min"].index(best_row["pooling"])
BEST_TRANSFORM   = test_transforms[BEST_EMB_TYPE][BEST_POOLING_IDX]

best_params = {
    "hidden_layer_sizes": best_row["best_hidden_layer_sizes"],  # string "(256, 128, 64)"
    "alpha":              best_row["best_alpha"],
    "learning_rate_init": best_row["best_lr_init"],
}

print(f"========================================================================")
print(f" MEJOR CONFIGURACIÓN ENCONTRADA ({TARGET_SCENARIO})")
print(f"========================================================================")
print(f"  Embedding:  {BEST_EMB_TYPE}")
print(f"  Pooling:    {best_row['pooling']}")
print(f"  Hiperparam: {best_params}")
print(f"------------------------------------------------------------------------")
print(f"  [MÉTRICAS GLOBALES DE LA CURVA]")
print(f"    AUPRG (PR-Gain AUC):     {best_row['auprg']:.4f}  (Métrica de selección)")
print(f"    E[FG1] (F1-Gain Esperado):{best_row['expected_f1_gain']:.4f}")
print(f"    PR AUC (Absoluto):       {best_row['pr_auc']:.4f}  |  PR AUC Lift: {best_row['pr_auc_lift']:.2f}x")
print(f"    ROC AUC:                 {best_row['roc_auc']:.4f}")
print(f"------------------------------------------------------------------------")
print(f"  [RENDIMIENTO EN PUNTOS OPERATIVOS SELECCIONADOS]")
print(f"    -> Con Umbral Fijo [0.50]:")
print(f"       Bal. Accuracy: {best_row['bal_accuracy_50']:.4f}  |  MCC: {best_row['mcc_50']:.4f}")
print(f"       PR-Gain Pct:   {best_row['pr_gain_50']:.4f}  |  F1-Gain: {best_row['f1_gain_50']:.4f}")
print(f"       Precision:     {best_row['precision_50']:.4f}  |  Recall: {best_row['recall_50']:.4f}")
print(f"    -> Con Umbral Optimizado [Umbral Promedio: {best_row['best_threshold']:.3f}]:")
print(f"       Bal. Accuracy: {best_row['bal_accuracy_opt']:.4f}  |  MCC: {best_row['mcc_opt']:.4f}")
print(f"       PR-Gain Pct:   {best_row['pr_gain_opt']:.4f}  |  F1-Gain: {best_row['f1_gain_opt']:.4f}")
print(f"       Precision:     {best_row['precision_opt']:.4f}  |  Recall: {best_row['recall_opt']:.4f}")
print(f"========================================================================")

 MEJOR CONFIGURACIÓN ENCONTRADA (C3)
  Embedding:  single_embeddings
  Pooling:    max
  Hiperparam: {'hidden_layer_sizes': '(256, 128, 64)', 'alpha': 0.0001, 'learning_rate_init': 0.0001}
------------------------------------------------------------------------
  [MÉTRICAS GLOBALES DE LA CURVA]
    AUPRG (PR-Gain AUC):     0.4060  (Métrica de selección)
    E[FG1] (F1-Gain Esperado):0.4646
    PR AUC (Absoluto):       0.4551  |  PR AUC Lift: 1.65x
    ROC AUC:                 0.6240
------------------------------------------------------------------------
  [RENDIMIENTO EN PUNTOS OPERATIVOS SELECCIONADOS]
    -> Con Umbral Fijo [0.50]:
       Bal. Accuracy: 0.5481  |  MCC: 0.0830
       PR-Gain Pct:   0.3086  |  F1-Gain: 0.3992
       Precision:     0.3253  |  Recall: 0.5685
    -> Con Umbral Optimizado [Umbral Promedio: 0.374]:
       Bal. Accuracy: 0.6568  |  MCC: 0.2923
       PR-Gain Pct:   0.4633  |  F1-Gain: 0.6274
       Precision:     0.4414  |  Recall: 0.8090


In [29]:
# ── 5. Sanity check con la mejor representación ──────────────────────────────
X_labeled, y_labeled, names_labeled = load_block_from_csv(
    OUTPUT_DIR,
    df_labeled,
    emb_type=BEST_EMB_TYPE,
    transforms=[BEST_TRANSFORM],
)
Xi = X_labeled[:, 0, :] if X_labeled.ndim == 3 else X_labeled

sanity_check_random_labels(Xi, y_labeled)

✅ Éxito: Se cargaron 962 muestras de las 962 del Excel.

=== Sanity Check: Labels Aleatorias (escenario C1) ===
    folds: total=5  válidos=5  NaN=0
  Repetición 1: balanced accuracy = 0.4937
    folds: total=5  válidos=5  NaN=0
  Repetición 2: balanced accuracy = 0.5018
    folds: total=5  válidos=5  NaN=0
  Repetición 3: balanced accuracy = 0.4840
  Media: 0.4932  (esperado ≈ 0.50)


In [30]:
# ── 6. Modelo final: entrenado con TODOS los datos etiquetados ───────────────
final_model = train_final_model(Xi, y_labeled, best_params)

Modelo final entrenado sobre 962 instancias con params: {'hidden_layer_sizes': '(256, 128, 64)', 'alpha': 0.0001, 'learning_rate_init': 0.0001}


In [31]:
# ── 7. Inferencia sobre datos dudosos ────────────────────────────────────────
# Cargar embeddings de los pares dudosos (zona Q2-Q3 del ranking topológico)
labeled_meta = df_labeled

X_unknown, _, names_unknown = load_block_from_csv(
    OUTPUT_DIR,
    df_unknown,
    emb_type=BEST_EMB_TYPE,
    transforms=[BEST_TRANSFORM],
)
Xi_unknown = X_unknown[:, 0, :] if X_unknown.ndim == 3 else X_unknown

# Metadatos de los dudosos (debe existir: sample_name, effector_group, protein_group)
unknown_meta = df_total[
    lambda d: d["Sample_name"].isin(names_unknown)
].copy()

# Metadatos de entrenamiento para clasificar nivel de confianza
train_meta = labeled_meta.copy()

df_inference = inference_to_csv(
    final_model,
    Xi_unknown,
    names_unknown,
    unknown_meta_df=unknown_meta,
    train_meta_df=train_meta,
    output_path=OUTPUT_RESULTS / "predictions_unknown.csv",
    eg_col="Effector_Group",
    pg_col="Protein_Group",
    e_id_col="Effector",  
    p_id_col="Protein",
)
display(df_inference.head(20))

✅ Éxito: Se cargaron 4504 muestras de las 4504 del Excel.
Predicciones guardadas en: /home/jovyan/TFG/CV_Kmeans_con_EspS_NleK_all_models/results/predictions_unknown.csv (4504 muestras)

Distribución de niveles de confianza (GRUPO):
confidence_level
C1    4504

Distribución de subniveles de confianza (INDIVIDUAL):
confidence_individual_sublevel
C1     1938
C2P    1200
C2E     966
C3      400


,sample,Effector,Protein,Is_Connected,Effector_Group,Protein_Group,Label,proba_interact,predicted_label,confidence_level,confidence_individual_sublevel
0,P60764_EspG,EspG,Rac3,False,Cluster_0,Cluster_3,-,0.999995,1,C1,C1
1,Q05144_EspG,EspG,Rac2,False,Cluster_0,Cluster_3,-,0.999993,1,C1,C1
2,P84096_EspG,EspG,Rhog,False,Cluster_0,Cluster_3,-,0.999976,1,C1,C1
3,Q9WVL2_Tir,Tir,Stat2,False,Cluster_2,Cluster_4,-,0.999972,1,C1,C2P
4,P97348_EspG,EspG,Rhod,False,Cluster_0,Cluster_3,-,0.999957,1,C1,C1
5,Q05144_EspV,EspV,Rac2,False,Cluster_0,Cluster_3,-,0.999955,1,C1,C2E
6,P60766_NleD,NleD,Cdc42,False,Cluster_6,Cluster_3,-,0.999951,1,C1,C1
7,Q9WVI9_NleB,NleB,Mapk8ip1,False,Cluster_0,Cluster_0,-,0.999946,1,C1,C1
8,P47811_NleD,NleD,Mapk14,True,Cluster_6,Cluster_3,-,0.999903,1,C1,C2P
9,P63085_EspG,EspG,Mapk1,False,Cluster_0,Cluster_3,-,0.999898,1,C1,C1


In [32]:
# ── 8. Resumen final por nivel de confianza ──────────────────────────────────
summary = (
    df_inference
    .groupby("confidence_level")
    .agg(
        n_pares=("sample", "count"),
        n_predicted_pos=("predicted_label", "sum"),
        proba_mean=("proba_interact", "mean"),
        proba_std=("proba_interact", "std"),
    )
    .reindex(["C1", "C2E", "C2P", "C3"])  # orden de mayor a menor confianza
    .reset_index()
)
print("\nResumen de inferencia por nivel de confianza:")
display(summary)


Resumen de inferencia por nivel de confianza:


,confidence_level,n_pares,n_predicted_pos,proba_mean,proba_std
0,C1,4504.0,1499.0,0.342421,0.387326
1,C2E,NaN,NaN,NaN,NaN
2,C2P,NaN,NaN,NaN,NaN
3,C3,NaN,NaN,NaN,NaN


In [33]:
# ── 8. Resumen final por nivel de confianza y subnivel ───────────────────────

# 1. Definimos el orden lógico de los niveles para que el resumen sea legible
level_order = ["C1", "C2E", "C2P", "C3"]

df_summary = df_inference.copy()

# 2. Convertimos a tipo Categorical para asegurar el orden correcto al agrupar
df_summary["confidence_level"] = pd.Categorical(
    df_summary["confidence_level"], categories=level_order, ordered=True
)
df_summary["confidence_individual_sublevel"] = pd.Categorical(
    df_summary["confidence_individual_sublevel"], categories=level_order, ordered=True
)

# 3. Agrupamos por ambas columnas
summary = (
    df_summary
    .groupby(["confidence_level", "confidence_individual_sublevel"], observed=False)
    .agg(
        n_pares=("sample", "count"),
        n_predicted_pos=("predicted_label", "sum"),
        proba_mean=("proba_interact", "mean"),
        proba_std=("proba_interact", "std"),
    )
    .reset_index()
    # Filtramos filas donde n_pares sea 0 para no saturar la tabla con combinaciones vacías
    .query("n_pares > 0") 
    .reset_index(drop=True)
)

print("\nResumen de inferencia por nivel (Grupo) y subnivel (Individual):")
display(summary)


Resumen de inferencia por nivel (Grupo) y subnivel (Individual):


,confidence_level,confidence_individual_sublevel,n_pares,n_predicted_pos,proba_mean,proba_std
0,C1,C1,1938,667,0.358006,0.390374
1,C1,C2E,966,253,0.275237,0.360634
2,C1,C2P,1200,448,0.374188,0.400987
3,C1,C3,400,131,0.333864,0.374631


# Repetición validación cruzada añadiendo subniveles en C1

In [25]:
# Generación de los nuevos splits
# Generador de splits Cx con subniveles C1 (debe estar en el mismo directorio o en PYTHONPATH)
sys.path.insert(0, str(Path("/home/jovyan/TFG/CV_Kmeans_con_EspS_NleK_all_models/").resolve()))
from generate_cx_splits_individual_sublevels import build_and_save_splits, load_splits

### Generación de splits

In [26]:
labeled_meta = df_labeled

# Generar (o regenerar) los splits exclusivos de subniveles C1
splits = build_and_save_splits(
    labeled_meta,
    output_dir=str("results/splits_subniveles"),
    effector_col="Effector",         # Nombre de tu columna individual
    protein_col="Protein",           # Nombre de tu columna individual
    label_col="Label",
    sample_col="Sample_name",
    min_train_ratio=0.50             # Mantiene el train por encima del 50% del tamaño original
)

print(f"\nDataset: {len(labeled_meta)} parejas  "
      f"({(labeled_meta.Label==1).sum()} pos, {(labeled_meta.Label==0).sum()} neg)")
print(f"Splits de C1 guardados en: {SPLITS_DIR}")

  REPORTE CV — SUBNIVELES EXHAUSTIVOS C1 CON FILTRO BICLASE ITERATIVO

Escenario C1 (1106 folds generados exitosamente)
───────────────────────────────────────────────────────────────────────────
  → Subnivel: C1_C2E (24 folds)
    Fold [C2E_eff_EspM                       ] train= 936 test=21 (pos=10, neg=11)
    Fold [C2E_eff_NleB                       ] train= 920 test=34 (pos=17, neg=17)
    Fold [C2E_eff_EspH                       ] train= 937 test=20 (pos=10, neg=10)
    Fold [C2E_eff_NleD                       ] train= 893 test=43 (pos=22, neg=21)
    Fold [C2E_eff_NleF                       ] train= 927 test=30 (pos=5, neg=25)
    [... y 19 folds más de este subnivel ...]
  → Subnivel: C1_C2P (120 folds)
    Fold [C2P_prot_Rhoc                      ] train= 951 test= 6 (pos=4, neg=2)
    Fold [C2P_prot_Ripk1                     ] train= 954 test= 3 (pos=2, neg=1)
    Fold [C2P_prot_Nfkb1                     ] train= 952 test= 5 (pos=3, neg=2)
    Fold [C2P_prot_Rac2             

In [29]:
# Comprobamos que está asegurándose de que en el train de cada fold se cumple el nivel estricto

# 1. Cargar la matriz de roles que se acaba de guardar
# (Asegúrate de apuntar a la ruta correcta de tu SPLITS_DIR)
SPLITS_DIR_SUBNIVELES = "results/splits_subniveles/"
folds = load_splits(SPLITS_DIR_SUBNIVELES, "C1")

# 2. Elegimos un fold cualquiera para auditar, por ejemplo uno de C2P
fold_a_comprobar = "C3_pair_NleD__Mapkapk2"
train_samples = folds[fold_a_comprobar]["train"]

# 3. Filtrar tu DataFrame original (labeled_meta) usando los nombres del train
# (Asumiendo que tu df original se llama labeled_meta)
df_train_audit = labeled_meta[labeled_meta["Sample_name"].isin(train_samples)]

# 4. Agrupar y verificar si algún individuo viola la regla biclase
eff_biclase_check = df_train_audit.groupby("Effector")["Label"].agg(['min', 'max'])
prot_biclase_check = df_train_audit.groupby("Protein")["Label"].agg(['min', 'max'])

# Si el filtro funcionó, 'min' debe ser 0 y 'max' debe ser 1 para TODOS
eff_fallos = eff_biclase_check[(eff_biclase_check['min'] == eff_biclase_check['max'])]
prot_fallos = prot_biclase_check[(prot_biclase_check['min'] == prot_biclase_check['max'])]

print(f"=== AUDITORÍA DEL FOLD: {fold_a_comprobar} ===")
print(f"Efectores individuales en Train que NO son biclase: {len(eff_fallos)}")
print(f"Proteínas individuales en Train que NO son biclase: {len(prot_fallos)}")

if len(eff_fallos) == 0 and len(prot_fallos) == 0:
    print("\n✅ ¡TODO PERFECTO! El filtro iterativo funciona de manera matemática estricta.")
else:
    print("\n❌ Hay algo raro, algunos individuos se han colgado con una sola clase.")

=== AUDITORÍA DEL FOLD: C3_pair_NleD__Mapkapk2 ===
Efectores individuales en Train que NO son biclase: 0
Proteínas individuales en Train que NO son biclase: 0

✅ ¡TODO PERFECTO! El filtro iterativo funciona de manera matemática estricta.


### Cross-validation

In [32]:
# ── Resolver Splitter Externo desde Carpeta C1 ────────────────────────────────
def get_outer_splitter_c1_sublevels(sublevel_prefix: str, sample_names: list, splits_dir) -> PrecomputedSplitter:
    """
    Carga el archivo splits_C1 y filtra los folds que pertenecen al subnivel activo.
    """
    from generate_cx_splits import load_splits
    all_c1_folds = load_splits(str(splits_dir), scenario="C1")
    
    filtered_folds = {
        k: v for k, v in all_c1_folds.items() 
        if k.startswith(sublevel_prefix)
    }
    
    if not filtered_folds:
        raise ValueError(f"No se encontraron folds con el prefijo '{sublevel_prefix}' en splits_C1_meta.json")
        
    return PrecomputedSplitter(filtered_folds, sample_names)

In [33]:
# ── Motor de Búsqueda de Hiperparámetros (Nested CV para MLP) ─────────────────
def hpm_search_nested_cv_sublevels(
    X, y,
    outer_splitter,
    sublevel_name: str,
    inner_cv: int = 5,
):
    """
    Nested CV optimizado para clasificadores MLP usando subniveles de C1.
    Aplica Media clásica para C1_C2E y Acumulación (Pooling) para C1_C2P y C1_C3.
    """
    param_grid = [
        {
            "mlpclassifier__hidden_layer_sizes": [(256, 128, 64), (128, 64, 32), (512, 256, 128)],
            "mlpclassifier__alpha":              [0.0001, 0.001, 0.01],
            "mlpclassifier__learning_rate_init": [0.001, 0.0001],
        }
    ]

    # Pipeline idéntico al de tu diseño para MLP desbalanceado
    pipeline = make_pipeline_imb(
        StandardScaler(),
        RandomOverSampler(random_state=42),
        MLPClassifier(
            activation='relu',
            solver='adam',
            max_iter=500,
            early_stopping=False, # Evita encoger aún más sets de train pequeños
            random_state=42,
        )
    )

    metrics_keys = (
        "test_bal_accuracy_50", "test_mcc_50", "test_precision_50", "test_recall_50", "test_pr_gain_50", "test_f1g_50",
        "test_bal_accuracy_opt", "test_mcc_opt", "test_precision_opt", "test_recall_opt", "test_pr_gain_opt", "test_f1g_opt",
        "test_best_threshold", "test_roc_auc", "test_pr_auc", "test_lift", "test_auprg", "test_expected_f1g"
    )
    
    results = {k: [] for k in metrics_keys}
    results["estimator"] = []
    results["fold_details"] = []

    # Vectores para acumulación (Pooling)
    y_true_acumulado = []
    y_proba_acumulado = []
    
    fold_ids = list(outer_splitter.folds.keys())
    is_pooling_scenario = sublevel_name in ["C1_C2P", "C1_C3"]

    for fold_id, (train_idx, test_idx) in zip(fold_ids, outer_splitter.split(X, y)):
        if len(test_idx) == 0:
            for k in metrics_keys: results[k].append(np.nan)
            results["estimator"].append(None)
            results["fold_details"].append({"fold_id": fold_id, "note": "vacío"})
            continue

        y_test = y[test_idx]
        X_train, y_train = X[train_idx], y[train_idx]
        X_test = X[test_idx]

        # Inner loop: StratifiedKFold limpio sin riesgo de fuga por el filtrado iterativo previo
        inner_splitter = StratifiedKFold(n_splits=inner_cv, shuffle=True, random_state=42)

        grid = GridSearchCV(
            pipeline, param_grid,
            cv=inner_splitter,
            scoring="average_precision", 
            n_jobs=-1,
            error_score=np.nan,
        )
        grid.fit(X_train, y_train)

        try:
            proba = grid.predict_proba(X_test)[:, 1]
            pi = float(y_test.sum() / len(y_test)) if len(y_test) > 0 else 0.0

            if is_pooling_scenario:
                y_true_acumulado.extend(y_test.tolist())
                y_proba_acumulado.extend(proba.tolist())

            # 1. EVALUACIÓN CON UMBRAL FIJO (0.50)
            pred_50 = (proba >= 0.5).astype(int)
            
            if len(y_test) == 1: # N=1 (Típico de C1_C3)
                bal_acc_50 = 1.0 if pred_50[0] == y_test[0] else 0.0
                mcc_50 = np.nan
                prec_50 = 1.0 if (pred_50[0] == 1 and y_test[0] == 1) else (0.0 if pred_50[0] == 1 else np.nan)
                rec_50  = 1.0 if (pred_50[0] == 1 and y_test[0] == 1) else (0.0 if y_test[0] == 1 else np.nan)
                f1_50   = 1.0 if (pred_50[0] == 1 and y_test[0] == 1) else 0.0
            else:
                bal_acc_50 = balanced_accuracy_score(y_test, pred_50)
                mcc_50     = matthews_corrcoef(y_test, pred_50)
                prec_50    = precision_score(y_test, pred_50, zero_division=0)
                rec_50     = recall_score(y_test, pred_50, zero_division=0)
                f1_50      = f1_score(y_test, pred_50, zero_division=0)
            
            pr_gain_50 = (prec_50 - pi) / ((1.0 - pi) * prec_50) if (not np.isnan(prec_50) and prec_50 > pi) else 0.0
            f1g_50     = (f1_50 - pi) / ((1.0 - pi) * f1_50) if f1_50 > pi else 0.0

            # 2. BARRIDO DE UMBRALES LOCALES (Solo si no es Pooling)
            if len(y_test) <= 1 or is_pooling_scenario:
                best_thresh, bal_acc_opt, mcc_opt, prec_opt, rec_opt, pr_gain_opt, f1g_opt = 0.5, bal_acc_50, mcc_50, prec_50, rec_50, pr_gain_50, f1g_50
                roc, pr, lift, auprg, expected_f1g = np.nan, np.nan, np.nan, np.nan, np.nan
            else:
                thresholds = np.linspace(0.01, 0.99, 100)
                best_f1, best_thresh = -1.0, 0.5
                for t in thresholds:
                    current_f1 = f1_score(y_test, (proba >= t).astype(int), zero_division=0)
                    if current_f1 > best_f1:
                        best_f1, best_thresh = current_f1, t

                pred_opt = (proba >= best_thresh).astype(int)
                bal_acc_opt = balanced_accuracy_score(y_test, pred_opt)
                mcc_opt     = matthews_corrcoef(y_test, pred_opt)
                prec_opt    = precision_score(y_test, pred_opt, zero_division=0)
                rec_opt     = recall_score(y_test, pred_opt, zero_division=0)
                pr_gain_opt = (prec_opt - pi) / ((1.0 - pi) * prec_opt) if prec_opt > pi else 0.0
                f1g_opt     = (best_f1 - pi) / ((1.0 - pi) * best_f1) if best_f1 > pi else 0.0

                roc  = roc_auc_score(y_test, proba) if len(np.unique(y_test)) > 1 else np.nan
                pr   = average_precision_score(y_test, proba) if len(np.unique(y_test)) > 1 else np.nan
                lift = pr / pi if (pi > 0 and not np.isnan(pr)) else np.nan
                
                # Integración geométrica para curvas completas (Solo C1_C2E)
                precisions, recalls, _ = precision_recall_curve(y_test, proba)
                with np.errstate(divide='ignore', invalid='ignore'):
                    prec_g_curve = (precisions - pi) / ((1.0 - pi) * precisions)
                    rec_g_curve = (recalls - pi) / ((1.0 - pi) * recalls)
                prec_g_curve = np.clip(np.nan_to_num(prec_g_curve, nan=0.0), 0.0, 1.0)
                rec_g_curve = np.clip(np.nan_to_num(rec_g_curve, nan=0.0), 0.0, 1.0)
                sort_idx = np.argsort(rec_g_curve)
                auprg = float(np.trapz(prec_g_curve[sort_idx], rec_g_curve[sort_idx]))
                
                zero_rec_indices = np.where(rec_g_curve[sort_idx] == 0.0)[0]
                y0 = float(prec_g_curve[sort_idx][zero_rec_indices[-1]]) if len(zero_rec_indices) > 0 else 0.0
                if y0 == 1.0:
                    expected_f1g = auprg / 2.0 + 0.25
                else:
                    expected_f1g = float((auprg / 2.0 + 0.25 - (pi * (1.0 - y0**2)) / 4.0) / (1.0 - pi * (1.0 - y0)))

            # Almacenar métricas en el diccionario
            results["test_bal_accuracy_50"].append(bal_acc_50)
            results["test_mcc_50"].append(mcc_50)
            results["test_precision_50"].append(prec_50)
            results["test_recall_50"].append(rec_50)
            results["test_pr_gain_50"].append(pr_gain_50)
            results["test_f1g_50"].append(f1g_50)
            results["test_bal_accuracy_opt"].append(bal_acc_opt)
            results["test_mcc_opt"].append(mcc_opt)
            results["test_precision_opt"].append(prec_opt)
            results["test_recall_opt"].append(rec_opt)
            results["test_pr_gain_opt"].append(pr_gain_opt)
            results["test_f1g_opt"].append(f1g_opt)
            results["test_best_threshold"].append(best_thresh)
            results["test_roc_auc"].append(roc)
            results["test_pr_auc"].append(pr)
            results["test_lift"].append(lift)
            results["test_auprg"].append(auprg)
            results["test_expected_f1g"].append(expected_f1g)
            results["estimator"].append(grid)
            
            # --- Extracción limpia de parámetros MLP para el reporte ---
            bp = grid.best_params_ if hasattr(grid, "best_params_") else {}
            hls_key   = next((k for k in bp if k.endswith("__hidden_layer_sizes")), None)
            alpha_key = next((k for k in bp if k.endswith("__alpha")), None)
            lr_key    = next((k for k in bp if k.endswith("__learning_rate_init")), None)

            results["fold_details"].append({
                "fold_id": fold_id, "n_test": len(test_idx),
                "n_test_pos": int(y_test.sum()), "n_test_neg": int(len(test_idx) - y_test.sum()),
                "roc_auc": round(roc, 4) if not np.isnan(roc) else np.nan,
                "pr_auc":  round(pr, 4) if not np.isnan(pr) else np.nan,
                "best_hidden_layer_sizes": str(bp.get(hls_key, "")) if hls_key else "",
                "best_alpha": bp.get(alpha_key, np.nan) if alpha_key else np.nan,
                "best_lr_init": bp.get(lr_key, np.nan) if lr_key else np.nan,
                "note": "ok"
            })

        except Exception as e:
            for k in metrics_keys: results[k].append(np.nan)
            results["estimator"].append(None)
            results["fold_details"].append({"fold_id": fold_id, "note": f"error: {str(e)}"})

    # =========================================================================
    # ── LOGICA DE POOLING CONSOLIDADO (MLP para C1_C2P y C1_C3) ──────────────
    # =========================================================================
    if is_pooling_scenario and len(y_true_acumulado) > 0 and len(np.unique(y_true_acumulado)) > 1:
        y_true_arr = np.array(y_true_acumulado)
        y_proba_arr = np.array(y_proba_acumulado)
        pi_global = float(np.mean(y_true_arr))
        
        g_roc  = float(roc_auc_score(y_true_arr, y_proba_arr))
        g_pr   = float(average_precision_score(y_true_arr, y_proba_arr))
        g_lift = float(g_pr / pi_global) if pi_global > 0 else np.nan
        
        # Integración geométrica global para el bloque acumulado completo
        precisions, recalls, _ = precision_recall_curve(y_true_arr, y_proba_arr)
        with np.errstate(divide='ignore', invalid='ignore'):
            prec_g_curve = (precisions - pi_global) / ((1.0 - pi_global) * precisions)
            rec_g_curve = (recalls - pi_global) / ((1.0 - pi_global) * recalls)
        prec_g_curve = np.clip(np.nan_to_num(prec_g_curve, nan=0.0), 0.0, 1.0)
        rec_g_curve = np.clip(np.nan_to_num(rec_g_curve, nan=0.0), 0.0, 1.0)
        sort_idx = np.argsort(rec_g_curve)
        g_auprg = float(np.trapz(prec_g_curve[sort_idx], rec_g_curve[sort_idx]))
        
        zero_rec_indices = np.where(rec_g_curve[sort_idx] == 0.0)[0]
        y0 = float(prec_g_curve[sort_idx][zero_rec_indices[-1]]) if len(zero_rec_indices) > 0 else 0.0
        g_f1g_e = float(g_auprg / 2.0 + 0.25) if y0 == 1.0 else float((g_auprg / 2.0 + 0.25 - (pi_global * (1.0 - y0**2)) / 4.0) / (1.0 - pi_global * (1.0 - y0)))

        # Búsqueda del umbral óptimo global sobre el pool
        thresholds = np.linspace(0.01, 0.99, 100)
        g_best_f1, g_best_thresh = -1.0, 0.5
        for t in thresholds:
            current_f1 = f1_score(y_true_arr, (y_proba_arr >= t).astype(int), zero_division=0)
            if current_f1 > g_best_f1:
                g_best_f1, g_best_thresh = current_f1, t
                
        pred_opt_g = (y_proba_arr >= g_best_thresh).astype(int)
        g_prec_opt = precision_score(y_true_arr, pred_opt_g, zero_division=0)
        g_rec_opt  = recall_score(y_true_arr, pred_opt_g, zero_division=0)
        g_bal_opt  = balanced_accuracy_score(y_true_arr, pred_opt_g)
        g_mcc_opt  = matthews_corrcoef(y_true_arr, pred_opt_g)
        g_prg_opt  = (g_prec_opt - pi_global) / ((1.0 - pi_global) * g_prec_opt) if g_prec_opt > pi_global else 0.0
        g_f1g_opt  = (g_best_f1 - pi_global) / ((1.0 - pi_global) * g_best_f1) if g_best_f1 > pi_global else 0.0
        
        # Inyección de los resultados fijos globales en cada fold del subnivel
        results["test_roc_auc"] = [g_roc] * len(fold_ids)
        results["test_pr_auc"] = [g_pr] * len(fold_ids)
        results["test_lift"] = [g_lift] * len(fold_ids)
        results["test_auprg"] = [g_auprg] * len(fold_ids)
        results["test_expected_f1g"] = [g_f1g_e] * len(fold_ids)
        results["test_best_threshold"] = [g_best_thresh] * len(fold_ids)
        results["test_bal_accuracy_opt"] = [g_bal_opt] * len(fold_ids)
        results["test_mcc_opt"] = [g_mcc_opt] * len(fold_ids)
        results["test_precision_opt"] = [g_prec_opt] * len(fold_ids)
        results["test_recall_opt"] = [g_rec_opt] * len(fold_ids)
        results["test_pr_gain_opt"] = [g_prg_opt] * len(fold_ids)
        results["test_f1g_opt"] = [g_f1g_opt] * len(fold_ids)
        
    return results

In [34]:
# ── Función de Control del Pipeline para Todos los Subniveles ─────────────────
def test_pooling_transforms_sublevels(transforms, labeled_dir, splits_dir, metadata_df: pd.DataFrame):
    """
    Orquesta la ejecución de la arquitectura MLP sobre los tres subniveles C1.
    """
    sublevels = {
        "C1_C2E": "C2E_",
        "C1_C2P": "C2P_",
        "C1_C3":  "C3_"
    }

    all_results = {}

    for sub_name, prefix in sublevels.items():
        print(f"\n{'#'*70}")
        print(f"#  SUBNIVEL INDIVIDUAL (MLP): {sub_name}")
        print(f"{'#'*70}")
        all_results[sub_name] = {}

        for emb_name, transform_list in transforms.items():
            print(f"\nEmbedding: {emb_name} ({len(transform_list)} poolings)")
            print("="*65)

            X_full, y_full, sample_names = load_block_from_csv(
                labeled_dir, 
                target_df=metadata_df[metadata_df['Label'].isin([0,1])], 
                emb_type=emb_name, 
                transforms=transform_list
            )

            if X_full.ndim == 3 and X_full.shape[0] != len(y_full):
                X_full = np.transpose(X_full, (1, 0, 2))

            res_emb = defaultdict(list)

            for i, _ in enumerate(transform_list):
                pooling_name = ["mean", "max", "min"][i] if i < 3 else f"pooling_{i}"
                print(f"\n  --- MLP | {emb_name} | {pooling_name} ---")

                Xi = X_full[:, i, :] if X_full.ndim == 3 else X_full
                outer_splitter = get_outer_splitter_c1_sublevels(prefix, sample_names, splits_dir)

                t0 = time.time()
                nested_res = hpm_search_nested_cv_sublevels(
                    Xi, y_full,
                    outer_splitter=outer_splitter,
                    sublevel_name=sub_name
                )
                elapsed = time.time() - t0

                # Captura estadística adaptativa
                roc    = np.array(nested_res["test_roc_auc"], dtype=float)
                pr     = np.array(nested_res["test_pr_auc"], dtype=float)
                auprg  = np.array(nested_res["test_auprg"], dtype=float)
                f1g_e  = np.array(nested_res["test_expected_f1g"], dtype=float)
                balopt = np.array(nested_res["test_bal_accuracy_opt"], dtype=float)
                mccopt = np.array(nested_res["test_mcc_opt"], dtype=float)
                th     = np.array(nested_res["test_best_threshold"], dtype=float)

                if sub_name in ["C1_C2P", "C1_C3"]:
                    mean_roc, mean_pr, mean_auprg, mean_f1g_e = roc[0], pr[0], auprg[0], f1g_e[0]
                    mean_bopt, mean_mopt, mean_th = balopt[0], mccopt[0], th[0]
                    print(f"    [Métricas Consolidadas por Pooling de Bloque]")
                else:
                    mean_roc, mean_pr, mean_auprg, mean_f1g_e = np.nanmean(roc), np.nanmean(pr), np.nanmean(auprg), np.nanmean(f1g_e)
                    mean_bopt, mean_mopt, mean_th = np.nanmean(balopt), np.nanmean(mccopt), np.nanmean(th)
                    print(f"    [Media de los {len(roc)} Folds Evaluados]")

                print(f"      ROC AUC: {mean_roc:.4f} | PR AUC: {mean_pr:.4f}")
                print(f"      AUPRG (PR-Gain): {mean_auprg:.4f} | E[FG1] (F1-Gain Esperado): {mean_f1g_e:.4f}")
                print(f"      Balanced Accuracy (Opt): {mean_bopt:.4f} | MCC (Opt): {mean_mopt:.4f} | Umbral: {mean_th:.3f}")
                print(f"      Tiempo total de cómputo del subnivel: {elapsed:.1f}s")

                # Almacenamiento indexado
                res_emb["pooling_name"].append(pooling_name)
                res_emb["cv_roc_auc"].append(mean_roc)
                res_emb["cv_pr_auc"].append(mean_pr)
                res_emb["cv_auprg"].append(mean_auprg)
                res_emb["cv_expected_f1g"].append(mean_f1g_e)
                res_emb["cv_bal_accuracy_opt"].append(mean_bopt)
                res_emb["cv_mcc_opt"].append(mean_mopt)
                res_emb["cv_best_threshold"].append(mean_th)
                res_emb["estimators"].append(nested_res["estimator"])

            all_results[sub_name][emb_name] = res_emb

    return all_results

In [35]:
# ── Lanzamiento de la Optimización y Entrenamiento MLP ───────────────────────
all_results_sublevels = test_pooling_transforms_sublevels(
    test_transforms,
    labeled_dir=OUTPUT_DIR,
    splits_dir=SPLITS_DIR_SUBNIVELES,
    metadata_df=labeled_meta
)


######################################################################
#  SUBNIVEL INDIVIDUAL (MLP): C1_C2E
######################################################################

Embedding: single_embeddings (3 poolings)
✅ Éxito: Se cargaron 962 muestras de las 962 del Excel.

  --- MLP | single_embeddings | mean ---


KeyboardInterrupt: 